In [1]:
import polars as pl
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import json
import numpy as np
from sklearn.cluster import KMeans
from datetime import datetime
from datetime import date
import re
from itertools import combinations
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
from scipy.sparse import coo_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfTransformer
from collections import defaultdict
import pyarrow.parquet as pq
import pyarrow as pa
from xgboost import XGBRegressor

In [13]:
item_df = pl.scan_parquet("data_full/sales_item/*.parquet")

purchase_history_df = pl.scan_parquet("data_full/sales_history/*.parquet")

user_df = pl.scan_parquet("data_full/sales_user/*.parquet")

In [14]:
print(item_df.explain(optimized=True))

Parquet SCAN [data_full\sales_item\sales_pers.item_chunk_0.parquet]
PROJECT */34 COLUMNS


In [15]:
print(user_df.explain(optimized=True))

Parquet SCAN [data_full\sales_user\sales_pers.user_chunk_0.parquet, ... 9 other sources]
PROJECT */18 COLUMNS


In [16]:
print(purchase_history_df.explain(optimized=True))

Parquet SCAN [data_full\sales_history\sales_pers.purchase_history_daily_chunk_0.parquet, ... 72 other sources]
PROJECT */16 COLUMNS


In [17]:
purchase_history_df[:1].collect()

timestamp,user_id,item_id,event_type,event_value,price,date_key,quantity,customer_id,created_date,updated_date,channel,payment,location,discount,is_deleted
i64,str,str,str,"decimal[38,4]","decimal[38,4]",i32,i32,i32,datetime[μs],datetime[μs],str,str,i32,"decimal[38,4]",bool
1735064221,"""ca12702ddf55acaa9fb767e10faaa6…","""7115000000004""","""Purchase""",1.0000,49000.0000,20241224,1,5254214,2024-12-24 18:17:01.027,2024-12-24 18:20:09.453,"""In-Store""","""VietQR""",656,0.0000,false


In [14]:
import polars as pl

result = (
    purchase_history_df
    .filter(
        (pl.col("created_date") >= pl.datetime(2025, 1, 1)) &
        (pl.col("created_date") <  pl.datetime(2025, 1, 31))
    )
    .collect()
)

print("Số dòng:", result.height)


Số dòng: 3298252


In [15]:
item_df[:1].collect()

p_id,item_id,price,category_l1_id,category_l1,category_l2_id,category_l2,category_l3_id,category_l3,category_id,category,description,brand,manufacturer,creation_timestamp,is_deleted,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,image_url,gender_target,age_group,item_type,gp,weight,color,size,origin,volume,material,sale_status,description_new
i32,str,"decimal[38,4]",i32,str,i32,str,i32,str,i32,str,str,str,str,i64,bool,datetime[μs],datetime[μs],i32,datetime[μs],str,str,str,str,str,"decimal[38,4]",f32,str,str,str,str,str,i32,str
17065,"""0502020000004""",99000.0000,1,"""Babycare""",35,"""Bình sữa, phụ kiện""",7050,"""Núm ty""",7058,"""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""",1333531544,false,2012-04-04 09:25:44.240,2025-08-18 09:59:19.847,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",36828.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Chi tiết sản phẩm …"


In [16]:
user_df[:1].collect()

customer_id,gender,location,province,membership,timestamp,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,region,location_name,install_app,install_date,district,user_id,is_deleted
i32,str,i32,str,str,i64,datetime[μs],datetime[μs],i32,datetime[μs],str,str,str,str,i64,str,str,bool
14732,"""Nam""",155,"""Hồ Chí Minh""","""Standard""",1306357911,2011-05-25 21:11:51.677,2025-07-07 15:33:10.201316,2,2025-07-16 11:54:29.816986,null,"""Đông Nam Bộ""","""HCM - Grand View Phú Mỹ Hưng""","""In-Store""",1306281600,"""7""","""e1e48206652bf8c279ff0206c69a80…",false


In [17]:
#laebl encoding cột giới tính 
item_df_lazy = item_df.lazy()
item_df_lazy = item_df_lazy.with_columns([
    pl.when(pl.col("gender_target") == "Bé Gái").then(0)
      .when(pl.col("gender_target") == "Bé Trai").then(1)
      .when(pl.col("gender_target") == "Không xác định").then(2)
      .when(pl.col("gender_target") == "Sơ sinh").then(3)
      .when(pl.col("gender_target") == "Unisex").then(4)
      .otherwise(-1)
      .alias("gender_target")
])
item_df = item_df_lazy.collect()

In [18]:
# Tạo LazyFrame từ item_df
item_df_lazy = item_df.lazy()

# 1️⃣ Tìm top 10 brand được mua nhiều nhất
top_brands = (
    item_df_lazy
    .group_by("brand")
    .agg(pl.count().alias("brand_count"))
    .sort("brand_count", descending=True)
    .head(10)
    .collect()  # Lấy danh sách brand top 10
)
top_brand_list = top_brands["brand"].to_list()

# 2️⃣ Label encoding: top 10 giữ nguyên, các brand khác thành "Other"
item_df_lazy = item_df_lazy.with_columns([
    pl.when(pl.col("brand").is_in(top_brand_list))
      .then(pl.col("brand"))
      .otherwise(pl.lit("Other"))
      .alias("brand_mapped")
])

# 3️⃣ Tạo mapping brand → số
unique_brands = item_df_lazy.select("brand_mapped").unique().collect()["brand_mapped"].to_list()
brand_mapping = {b: i for i, b in enumerate(unique_brands)}

# 4️⃣ Áp dụng mapping
item_df_lazy = item_df_lazy.with_columns([
    pl.col("brand_mapped").map_elements(lambda x: brand_mapping.get(x, -1)).alias("brand")
]).drop("brand_mapped")


# 6️⃣ Khi cần, collect về DataFrame bình thường
item_df = item_df_lazy.collect()

print(item_df.select(["brand"]).head(15))
print(f"Tổng số brand sau khi mã hóa: {len(unique_brands)}")
print("Mapping:", brand_mapping)


shape: (15, 1)
┌───────┐
│ brand │
│ ---   │
│ i64   │
╞═══════╡
│ 2     │
│ 7     │
│ 0     │
│ 2     │
│ 2     │
│ …     │
│ 9     │
│ 2     │
│ 2     │
│ 2     │
│ 2     │
└───────┘
Tổng số brand sau khi mã hóa: 11
Mapping: {'Thương hiệu khác': 0, 'Mesuca': 1, 'Other': 2, 'TOYCITY': 3, 'ConCung Good': 4, 'Không xác định': 5, 'Pigeon': 6, 'Con Cưng': 7, 'Animo': 8, 'CF (ConCung Fashion)': 9, 'Nous': 10}


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_26976\1428647967.py:8: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("brand_count"))


In [19]:
# 1️⃣ Tạo LazyFrame
purchase_lazy = purchase_history_df.lazy()
# 2️⃣ Lấy danh sách các channel unique
unique_channels = (
    purchase_lazy.select("channel")
    .unique()
    .collect()["channel"]
    .to_list()
)
# 3️⃣ Tạo mapping
channel_mapping = {ch: i for i, ch in enumerate(unique_channels)}
# 4️⃣  dụng mapping (Lazy)
purchase_lazy = purchase_lazy.with_columns([
    pl.col("channel").map_elements(lambda x: channel_mapping.get(x, -1)).alias("channel")
])
# 6️⃣ Khi cần collect về DataFrame bình thường
purchase_history_df = purchase_lazy.collect()
# 7️⃣ In ra kết quả
print("Mapping:", channel_mapping)
print(purchase_history_df.select("channel").head(10))

Mapping: {'iOS': 0, 'Wholesale': 1, 'SPE': 2, 'Không xác định': 3, 'Call': 4, 'Android': 5, 'TKS': 6, 'CRM Partner': 7, 'Web': 8, 'Chat': 9, 'In-Store': 10}
shape: (10, 1)
┌─────────┐
│ channel │
│ ---     │
│ i64     │
╞═════════╡
│ 10      │
│ 10      │
│ 10      │
│ 10      │
│ 5       │
│ 10      │
│ 10      │
│ 10      │
│ 10      │
│ 10      │
└─────────┘


In [20]:
# 1️⃣ Tạo LazyFrame
purchase_lazy = purchase_history_df.lazy()
# 2️⃣ Lấy danh sách các payment unique
unique_payments = (
    purchase_lazy.select("payment")
    .unique()
    .collect()["payment"]
    .to_list()
)
# 3️⃣ Tạo mapping
payment_mapping = {p: i for i, p in enumerate(unique_payments)}
# 4️⃣ Áp dụng mapping (Lazy)
purchase_lazy = purchase_lazy.with_columns([
    pl.col("payment").map_elements(lambda x: payment_mapping.get(x, -1)).alias("payment")
])
# 6️⃣ Khi cần collect về DataFrame bình thường
purchase_history_df = purchase_lazy.collect()
# 7️⃣ In kết quả
print("Mapping:", payment_mapping)
print(purchase_history_df.select("payment").head(10))

Mapping: {'VietQR': 0, 'Tiền mặt': 1, 'ShopeePay': 2, 'Không xác định': 3, 'ZaloPay': 4, 'VNPay': 5, 'Chuyển khoản': 6, 'Cà thẻ': 7, 'MoMo': 8, 'Kredivo': 9}
shape: (10, 1)
┌─────────┐
│ payment │
│ ---     │
│ i64     │
╞═════════╡
│ 0       │
│ 1       │
│ 8       │
│ 8       │
│ 8       │
│ 1       │
│ 1       │
│ 0       │
│ 0       │
│ 5       │
└─────────┘


In [21]:
# 1️⃣ Tạo LazyFrame từ user_df
user_lazy = user_df.lazy()
# 2️⃣ Lấy danh sách các giá trị gender duy nhất
unique_genders = (
    user_lazy.select("gender")
    .unique()
    .collect()["gender"]  # collect để lấy Series
    .to_list()
)
# 3️⃣ Tạo mapping
gender_mapping = {g: i for i, g in enumerate(unique_genders)}
# 4️⃣ In mapping ra
print("Mapping gender → số:")
for k, v in gender_mapping.items():
    print(f"{k} -> {v}")
# 5️⃣ Áp dụng mapping trên LazyFrame
user_lazy = user_lazy.with_columns([
    pl.col("gender").map_elements(lambda x: gender_mapping.get(x, -1)).alias("gender")
])
# 6️⃣ Khi cần, collect về DataFrame bình thường
user_df = user_lazy.collect()
# 7️⃣ Kiểm tra 10 dòng đầu
print(user_df.select("gender").head(10))

Mapping gender → số:
Khác -> 0
Nữ -> 1
Nam -> 2
shape: (10, 1)
┌────────┐
│ gender │
│ ---    │
│ i64    │
╞════════╡
│ 2      │
│ 1      │
│ 1      │
│ 1      │
│ 1      │
│ 1      │
│ 1      │
│ 1      │
│ 1      │
│ 1      │
└────────┘


In [22]:
# 1️⃣ Tạo LazyFrame từ user_df
user_lazy = user_df.lazy()
# 2️⃣ Lấy danh sách các membership duy nhất
unique_memberships = (
    user_lazy.select("membership")
    .unique()
    .collect()["membership"]  # collect để lấy giá trị ra Series
    .to_list()
)
# 3️⃣ Tạo mapping
membership_mapping = {m: i for i, m in enumerate(unique_memberships)}
print("Membership mapping:", membership_mapping)
# 4️⃣ Áp dụng mapping trên LazyFrame
user_lazy = user_lazy.with_columns([
    pl.col("membership").map_elements(lambda x: membership_mapping.get(x, -1)).alias("membership")
])
# 5️⃣ Khi cần, collect về DataFrame bình thường
user_df = user_lazy.collect()
# 6️⃣ In kết quả
print(user_df.select("membership").limit(10))

Membership mapping: {'Diamond': 0, 'Standard': 1, 'Gold': 2}
shape: (10, 1)
┌────────────┐
│ membership │
│ ---        │
│ i64        │
╞════════════╡
│ 1          │
│ 1          │
│ 1          │
│ 1          │
│ 1          │
│ 1          │
│ 1          │
│ 1          │
│ 1          │
│ 1          │
└────────────┘


In [23]:
# 1️⃣ Tạo LazyFrame từ user_df
user_lazy = user_df.lazy()
# 2️⃣ Lấy danh sách các giá trị region duy nhất
unique_regions = (
    user_lazy.select("region")
    .unique()
    .collect()["region"]  # collect để lấy Series
    .to_list()
)
# 3️⃣ Tạo mapping
region_mapping = {r: i for i, r in enumerate(unique_regions)}
# 4️⃣ In mapping ra
print("Mapping region → số:")
for k, v in region_mapping.items():
    print(f"{k} -> {v}")
# 5️⃣ Áp dụng mapping trên LazyFrame
user_lazy = user_lazy.with_columns([
    pl.col("region").map_elements(lambda x: region_mapping.get(x, -1)).alias("region")
])
# 6️⃣ Khi cần, collect về DataFrame bình thường
user_df = user_lazy.collect()
# 7️⃣ Kiểm tra 10 dòng đầu
print(user_df.select("region").head(10))

Mapping region → số:
Trung du và miền núi phía Bắc -> 0
Đồng bằng sông Cửu Long -> 1
Duyên hải Nam Trung Bộ -> 2
Đông Nam Bộ -> 3
Bắc Trung Bộ -> 4
Tây Nguyên -> 5
Đồng bằng sông Hồng -> 6
Duyên hải Bắc Bộ -> 7
shape: (10, 1)
┌────────┐
│ region │
│ ---    │
│ i64    │
╞════════╡
│ 3      │
│ 3      │
│ 1      │
│ 3      │
│ 3      │
│ 3      │
│ 3      │
│ 1      │
│ 3      │
│ 3      │
└────────┘


In [24]:
# 1️⃣ Tạo LazyFrame từ user_df
user_lazy = user_df.lazy()
# 2️⃣ Lấy danh sách các giá trị install_app duy nhất
unique_install_apps = (
    user_lazy.select("install_app")
    .unique()
    .collect()["install_app"]
    .to_list()
)
# 3️⃣ Tạo mapping
install_app_mapping = {app: i for i, app in enumerate(unique_install_apps)}
# 4️⃣ In mapping ra
print("Mapping install_app → số:")
for k, v in install_app_mapping.items():
    print(f"{k} -> {v}")
# 5️⃣ Áp dụng mapping trên LazyFrame
user_lazy = user_lazy.with_columns([
    pl.col("install_app")
      .map_elements(lambda x: install_app_mapping.get(x, -1), return_dtype=pl.Int32)
      .alias("install_app")
])
# 6️⃣ Khi cần, collect về DataFrame bình thường
user_df = user_lazy.collect()
# 7️⃣ Kiểm tra 10 dòng đầu
print(user_df.select("install_app").head(10))

Mapping install_app → số:
Web -> 0
CRM Partner -> 1
Android -> 2
In-Store -> 3
iOS -> 4
Chat -> 5
LZD -> 6
SPE -> 7
Wholesale -> 8
Call -> 9
Không xác định -> 10
shape: (10, 1)
┌─────────────┐
│ install_app │
│ ---         │
│ i32         │
╞═════════════╡
│ 3           │
│ 3           │
│ 3           │
│ 3           │
│ 3           │
│ 3           │
│ 3           │
│ 3           │
│ 3           │
│ 3           │
└─────────────┘


In [25]:
# 1️⃣ Tạo LazyFrame từ user_df
user_lazy = user_df.lazy()
# 2️⃣ Chuẩn hóa province
user_lazy = user_lazy.with_columns(
    pl.col("province")
      .str.strip_chars()
      .str.replace("Tỉnh ", "")
      .str.replace("Thành phố ", "")
      .str.replace("Thành Phố ", "")
      .alias("province")
)
# 3️⃣ Lấy danh sách tỉnh duy nhất, sắp xếp và tạo mapping
province_list = (
    user_lazy.select("province")
    .unique()
    .sort("province")
    .collect()
    .to_series()
    .to_list()
)
province_mapping = {prov: i for i, prov in enumerate(province_list)}
# 4️⃣ Áp dụng mapping trên LazyFrame
user_lazy = user_lazy.with_columns(
    pl.col("province").map_elements(lambda x: province_mapping.get(x, -1), return_dtype=pl.Int32).alias("province")
)
# 5️⃣ Collect về DataFrame bình thường
user_df = user_lazy.collect()
# 6️⃣ Kiểm tra 10 dòng đầu
print(user_df.select("province").head(10))

shape: (10, 1)
┌──────────┐
│ province │
│ ---      │
│ i32      │
╞══════════╡
│ 24       │
│ 24       │
│ 10       │
│ 24       │
│ 24       │
│ 24       │
│ 24       │
│ 13       │
│ 24       │
│ 24       │
└──────────┘


In [26]:
# 1️⃣ Tạo LazyFrame
purchase_lazy = purchase_history_df.lazy()
# 2️⃣ Tính quantile 0.99
q99_df = purchase_lazy.select(pl.col("price").quantile(0.99)).collect()
p99 = q99_df.to_numpy()[0][0]
print(f"Quantile 0.99 của price: {p99}")
# 3️⃣ Cắt outlier và log1p trên LazyFrame
purchase_lazy = purchase_lazy.with_columns(
    pl.when(pl.col("price") > p99)
      .then(p99)
      .otherwise(pl.col("price"))
      .log1p()
      .alias("price")
)
# 4️⃣ Collect về DataFrame bình thường
purchase_history_df = purchase_lazy.collect()
# 5️⃣ Kiểm tra 10 dòng đầu
print(purchase_history_df.select(["price"]).head(10))

Quantile 0.99 của price: 805000.0
shape: (10, 1)
┌───────────┐
│ price     │
│ ---       │
│ f64       │
╞═══════════╡
│ 10.799596 │
│ 11.141876 │
│ 11.225257 │
│ 10.976799 │
│ 11.396403 │
│ 11.133678 │
│ 12.594734 │
│ 12.301387 │
│ 12.594734 │
│ 9.615872  │
└───────────┘


In [ ]:
# ------------------------------------------------
# 1. FILTER PURCHASE + CLEAN DATE
# ------------------------------------------------
df = (
    purchase_history_df
    .filter(
        (pl.col("event_type") == "Purchase") &
        (pl.col("is_deleted") == False) &
        (pl.col("created_date").is_not_null())
    )
    .select([
        "customer_id",
        "item_id",
        "quantity",
        "price",
        "discount",
        "created_date"
    ])
)

now = pl.lit(pd.Timestamp.now())

df = df.with_columns(
    (now - pl.col("created_date"))
        .dt.total_days()
        .clip(lower_bound=0)
        .alias("days_ago")
)

df = df.with_columns(
    (
        (pl.col("quantity") * pl.col("price") - pl.col("discount"))
        * pl.col("days_ago").map_elements(lambda x: np.exp(-0.01 * x))
    ).alias("score")
)

# ------------------------------------------------
# 2. AGG USER–ITEM
# ------------------------------------------------
ui = (
    df.group_by(["customer_id", "item_id"])
      .agg(pl.col("score").sum())
      .filter(pl.col("score").is_finite())
)

ui_pd = ui.to_pandas()

# ------------------------------------------------
# 3. BUILD MATRIX
# ------------------------------------------------
user_map = {u: i for i, u in enumerate(ui_pd["customer_id"].unique())}
item_map = {i: j for j, i in enumerate(ui_pd["item_id"].unique())}
inv_item_map = {v: k for k, v in item_map.items()}

ui_pd["u_idx"] = ui_pd["customer_id"].map(user_map)
ui_pd["i_idx"] = ui_pd["item_id"].map(item_map)

matrix = csr_matrix(
    (ui_pd["score"], (ui_pd["u_idx"], ui_pd["i_idx"]))
)

item_user_matrix = matrix.T  # (num_items, num_users)

# ------------------------------------------------
# 4. ITEM-CF (COSINE)
# ------------------------------------------------
item_similarity_cf = cosine_similarity(
    item_user_matrix,
    dense_output=False
)

item_importance_cf = np.array(
    item_similarity_cf.sum(axis=1)
).ravel()

top_cf = (
    pd.DataFrame({
        "item_id": [inv_item_map[i] for i in range(len(item_importance_cf))],
        "score_cf": item_importance_cf
    })
    .sort_values("score_cf", ascending=False)
    .head(40)
)

# ------------------------------------------------
# 5. PMI / PPMI
# ------------------------------------------------
binary_matrix = item_user_matrix.copy()
binary_matrix.data = np.ones_like(binary_matrix.data)

# Co-occurrence
C = (binary_matrix @ binary_matrix.T).tocoo()
C.setdiag(0)

# Frequencies
item_freq = np.array(C.sum(axis=1)).ravel()
total = item_freq.sum()
item_freq[item_freq == 0] = 1e-9

# ==== PMI tính trực tiếp trên non-zero entries ====
rows = C.row
cols = C.col
data = C.data

pmi_data = np.log((data * total) / (item_freq[rows] * item_freq[cols]))
pmi_data[pmi_data < 0] = 0  # PPMI

PMI = coo_matrix((pmi_data, (rows, cols)), shape=C.shape)

# Item importance
item_importance_pmi = np.array(PMI.sum(axis=1)).ravel()

top_pmi = (
    pd.DataFrame({
        "item_id": [inv_item_map[i] for i in range(len(item_importance_pmi))],
        "score_pmi": item_importance_pmi
    })
    .sort_values("score_pmi", ascending=False)
    .head(40)
)

# ------------------------------------------------
# 6. TOP 500 POPULAR
# ------------------------------------------------
top_popularity = (
    purchase_history_df
    .filter(
        (pl.col("event_type") == "Purchase") &
        (pl.col("is_deleted") == False)
    )
    .group_by("item_id")
    .agg(pl.count().alias("num_transactions"))
    .sort("num_transactions", descending=True)
    .head(30)
    .select("item_id")
    .to_pandas()
)

# ------------------------------------------------
# 7. FINAL CANDIDATE
# ------------------------------------------------
final_candidate_items = (
    pd.concat([
        top_cf[["item_id"]],
        top_pmi[["item_id"]],
        top_popularity[["item_id"]]
    ])
    .drop_duplicates()
    .reset_index(drop=True)
)

final_candidate_items.to_parquet(
    "final_candidate_items.parquet",
    index=False
)
df_final = pl.read_parquet("final_candidate_items.parquet")
print(f"Number of items in final file: {df_final.height}")

In [27]:
# ------------------------------------------------
# 1. Chọn giao dịch hợp lệ (Purchase, chưa xoá, có ngày)
# ------------------------------------------------
df = purchase_history_df.filter(
    (pl.col("event_type") == "Purchase") &
    (pl.col("is_deleted") == False) &
    (pl.col("created_date").is_not_null())
)

# Nếu created_date là string → parse
if df.schema["created_date"] == pl.Utf8:
    df = df.with_columns(
        pl.col("created_date").str.strptime(pl.Datetime, strict=False)
    )

# ------------------------------------------------
# 2. Lọc giao dịch trong năm 2024–2025
# ------------------------------------------------
df_2024_2025 = df.filter(
    pl.col("created_date").dt.year().is_in([2024, 2025])
)

# ------------------------------------------------
# 3. Khách hàng có ≥ 2 giao dịch
# ------------------------------------------------
final_active_customers = (
    df_2024_2025
    .group_by("customer_id")
    .agg(pl.len().alias("num_orders"))
    .filter(pl.col("num_orders") >= 2)
    .select("customer_id")
)

print(
    "Số khách hàng có ≥2 giao dịch trong năm 2024–2025:",
    final_active_customers.height
)

# ------------------------------------------------
# 4. Lưu parquet
# ------------------------------------------------
final_active_customers.write_parquet("customer_list.parquet")



Số khách hàng có ≥2 giao dịch trong năm 2024–2025: 1924932


In [46]:
import polars as pl
import math

# Load dữ liệu
customers = pl.read_parquet("customer_list.parquet").select("customer_id")
items = pl.read_parquet("final_candidate_items.parquet").select("item_id")

n_items = items.height
target_rows_per_chunk = 1_000_000

# Số customer mỗi chunk ≈ 1M / n_items
customers_per_chunk = max(1, target_rows_per_chunk // n_items)

print("Items:", n_items)
print("Customers per chunk:", customers_per_chunk)

total_customers = customers.height
chunk_idx = 0
total_rows = 0

for start in range(0, total_customers, customers_per_chunk):
    cust_chunk = customers.slice(start, customers_per_chunk)

    joined = cust_chunk.join(items, how="cross")
    rows = joined.height
    total_rows += rows

    out_file = f"customer_item_keys_chunk_{chunk_idx}.parquet"
    joined.write_parquet(out_file)

    print(f"Saved {out_file} with {rows:,} rows")

    chunk_idx += 1

print(f"Total rows generated: {total_rows:,}")


Items: 97
Customers per chunk: 10309
Saved customer_item_keys_chunk_0.parquet with 999,973 rows
Saved customer_item_keys_chunk_1.parquet with 999,973 rows
Saved customer_item_keys_chunk_2.parquet with 999,973 rows
Saved customer_item_keys_chunk_3.parquet with 999,973 rows
Saved customer_item_keys_chunk_4.parquet with 999,973 rows
Saved customer_item_keys_chunk_5.parquet with 999,973 rows
Saved customer_item_keys_chunk_6.parquet with 999,973 rows
Saved customer_item_keys_chunk_7.parquet with 999,973 rows
Saved customer_item_keys_chunk_8.parquet with 999,973 rows
Saved customer_item_keys_chunk_9.parquet with 999,973 rows
Saved customer_item_keys_chunk_10.parquet with 999,973 rows
Saved customer_item_keys_chunk_11.parquet with 999,973 rows
Saved customer_item_keys_chunk_12.parquet with 999,973 rows
Saved customer_item_keys_chunk_13.parquet with 999,973 rows
Saved customer_item_keys_chunk_14.parquet with 999,973 rows
Saved customer_item_keys_chunk_15.parquet with 999,973 rows
Saved custome

In [20]:
begin_hist = datetime.strptime('2024-01-01', '%Y-%m-%d')
end_hist   = datetime.strptime('2024-11-30', '%Y-%m-%d')
begin_recent = datetime.strptime('2024-05-01', '%Y-%m-%d')
end_recent   = datetime.strptime('2025-01-31', '%Y-%m-%d')
#begin_val = datetime.strptime('2024-12-01', '%Y-%m-%d')
#end_val   = datetime.strptime('2024-12-31', '%Y-%m-%d')
#begin_test =datetime.strptime('2025-01-01', '%Y-%m-%d')
#end_test=datetime.strptime('2025-01-31', '%Y-%m-%d')
train_history = purchase_history_df.lazy().filter(
    (pl.col("created_date") >= begin_hist) & (pl.col("created_date") <= end_hist)
)
train_recent = purchase_history_df.lazy().filter(
    (pl.col("created_date") >= begin_recent) & (pl.col("created_date") <= end_recent)
)
#val = purchase_history_df.lazy().filter(
#    (pl.col("created_date") >= begin_val) & (pl.col("created_date") <= end_val)
#)


In [21]:
# hàm chuyển các nhóm tuổi về month 

def normalize_and_split_age_group(age_str: str):
    if not age_str or age_str.strip() == "" or age_str.upper() == "KHÔNG XÁC ĐỊNH":
        return []

    s = age_str.upper().replace(" ", "")

    def to_months(value: str) -> int:
        years = months = 0
        y_match = re.search(r'(\d+)Y', value)
        m_match = re.search(r'(\d+)M', value)
        if y_match: years = int(y_match.group(1))
        if m_match: months = int(m_match.group(1))
        if not y_match and not m_match and value.isdigit(): months = int(value)
        return years * 12 + months

    parts = re.split(r'[-–]', s)
    if len(parts) == 2:
        start = to_months(parts[0])
        end = to_months(parts[1])
    elif "TỪ" in s or "TRÊN" in s:
        m = re.search(r'(\d+[MY])', s).group(1)
        start = to_months(m)
        end = start + 60
    else:
        start = to_months(s)
        end = start

    return [f"{m}M" for m in range(start, end + 1, 6)]  # mỗi 6 tháng


In [22]:
def compute_X1(train_history, item):
    df = (
        train_history.lazy()
        .join(item.lazy(), on="item_id", how="inner")
        .group_by(["customer_id", "item_id"])
        .agg((pl.col("brand") == pl.col("brand")).sum().alias("X_1"))
    )
    return df

In [23]:
feature_X1 =compute_X1(train_history, item_df)
feature_X1.collect().write_parquet("feature_X1_train.parquet", compression="snappy", use_pyarrow=True)

In [24]:
feature_X1=compute_X1(train_recent, item_df)
feature_X1.collect().write_parquet("feature_X1_val.parquet", compression="snappy", use_pyarrow=True)

In [25]:
def compute_X2(train_history, item):
    df = (
        train_history.lazy()
        .join(item.lazy(), on="item_id", how="inner")
        .group_by(["customer_id", "item_id"])
        .agg((pl.col("age_group") == pl.col("age_group")).sum().alias("X_2"))
    )
    return df

In [26]:
feature_X2 =compute_X2(train_history, item_df)
feature_X2.collect().write_parquet("feature_X2_train.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [27]:
feature_X2=compute_X2(train_recent, item_df)
feature_X2.collect().write_parquet("feature_X2_val.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [28]:
def compute_X3(train_history, item):
    df = (
        train_history.lazy()
        .join(item.lazy(), on="item_id", how="inner")
        .group_by(["customer_id", "item_id"])
        .agg((pl.col("category") == pl.col("category")).sum().alias("X_3"))
    )
    return df

In [29]:
feature_X3 =compute_X3(train_history, item_df)
feature_X3.collect().write_parquet("feature_X3_train.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [30]:
feature_X3=compute_X3(train_recent, item_df)
feature_X3.collect().write_parquet("feature_X3_val.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [22]:
# đếm số lần item được mua chung với các item khác 
def compute_X4(train_history):
    """
    Tính feature X_4: số lần item được mua chung với các item khác (bỏ customer)
    
    Args:
        train_history: pl.DataFrame, chứa item_id, created_date
    
    Returns:
        pl.DataFrame: item_id, X_4
    """
    
    # Gom các item theo đơn (created_date + giả sử mỗi row = 1 order)
    orders = train_history.lazy().group_by("created_date") \
                           .agg(pl.col("item_id").sort().alias("items"))
    
    # Explode để tạo cặp item trong cùng đơn
    orders_exploded = orders.explode("items").rename({"items": "item_1"})
    
    cooc_df = (
        orders_exploded.join(orders_exploded, on="created_date", how="inner")
        .filter(pl.col("item_1") < pl.col("item_1_right"))
        .group_by(["item_1", "item_1_right"])
        .agg(pl.count().alias("cooc_count"))
        .rename({"item_1_right": "item_2"})
    )
    
    # Tính X_4 cho từng item
    X4_df = (
        cooc_df.lazy()
        .select([
            pl.col("item_1").alias("item_id"), 
            pl.col("cooc_count")
        ])
        .group_by("item_id")
        .agg(pl.col("cooc_count").sum().alias("X_4"))
        .collect()
    )
    
    return X4_df


In [23]:
feature_X4_items = compute_X4(train_history)
feature_X4_items.write_parquet("feature_X4_train.parquet", compression="snappy", use_pyarrow=True)
print("finish")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10376\1625565777.py:24: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("cooc_count"))


finish


In [24]:
feature_X4=compute_X4(train_recent)
feature_X4.write_parquet("feature_X4_val.parquet", compression="snappy", use_pyarrow=True)
print("finish")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10376\1625565777.py:24: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("cooc_count"))


finish


In [25]:
# xác định nhóm tuổi của item theo month 
def compute_X5(item_df: pl.DataFrame):
    """
    Tạo feature X_5 cho item:
    - Gộp tất cả các nhóm tuổi (6M, 12M, …, 120M) thành 1 cột tổng.
    - Trả về DataFrame chỉ có item_id và X_5.
    """

    # 1️⃣ LazyFrame với cột X_5 dạng list
    lf = item_df.lazy().with_columns(
        pl.col("age_group")
          .map_elements(normalize_and_split_age_group, return_dtype=pl.List(pl.Utf8))
          .alias("X_5")
    )

    # 2️⃣ Collect về eager
    df = lf.collect()

    # 3️⃣ Explode list -> mỗi row 1 mốc tuổi
    df = df.explode("X_5")

    # 4️⃣ Tạo multi-hot: cột X5_6M … X5_120M
    all_months = [f"{m}M" for m in range(6, 121, 6)]
    for m in all_months:
        df = df.with_columns(
            (pl.col("X_5") == m).cast(pl.UInt8).alias(f"X5_{m}")
        )

    # 5️⃣ Group lại theo item_id, max từng cột
    df = df.group_by("item_id").agg([pl.max(f"X5_{m}").alias(f"X5_{m}") for m in all_months])

    # 6️⃣ Gộp 20 cột thành 1 cột tổng
    df = df.with_columns(
    sum([pl.col(f"X5_{m}") for m in all_months]).alias("X_5")
     ).drop([f"X5_{m}" for m in all_months])  # xóa các cột cũ đúng tên

    # 7️⃣ Lưu parquet
    df.write_parquet("feature_X5.parquet", compression="snappy", use_pyarrow=True)

    return df

In [26]:
feature_X5=compute_X5(item_df)
feature_X5.write_parquet("feature_X5_train.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [27]:
feature_X5=compute_X5(item_df)
feature_X5.write_parquet("feature_X5_val.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [28]:
def compute_X6(train_history: pl.LazyFrame) -> pl.DataFrame:
    df = train_history.select(
        "customer_id",
        (
            pl.col("price").cast(pl.Decimal(38,4)) *
            pl.col("quantity").cast(pl.Decimal(38,4))
        ).alias("total_amount")
    )

    return (
        df.group_by("customer_id")
          .agg(pl.col("total_amount").mean().alias("X_6"))
          .collect()
    )



In [29]:
feature_X6 = compute_X6(train_history)
feature_X6.write_parquet("feature_X6_train.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [30]:
feature_X6 = compute_X6(train_recent)
feature_X6.write_parquet("feature_X6_val.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [31]:
def compute_X7_X8_X9(train_history: pl.LazyFrame, item_df: pl.LazyFrame):
    """
    Tính các feature:
        X_7: tổng số đơn hàng
        X_8: số item trung bình mỗi đơn hàng
        X_9: số brand khác nhau mà khách hàng đã từng mua (dựa trên item_df)

    Args:
        train_history: pl.LazyFrame, chứa customer_id, item_id, created_date
        item_df: pl.LazyFrame, chứa item_id và brand

    Returns:
        Tuple[pl.DataFrame]: X7, X8, X9
    """

    # -------------------------------
    # X7: tổng số đơn hàng
    # -------------------------------
    X7 = (
        train_history
        .group_by("customer_id")
        .agg(pl.count("created_date").alias("X_7"))
        .collect()
    )

    # -------------------------------
    # X8: số item trung bình mỗi đơn hàng
    # -------------------------------
    orders = (
        train_history
        .group_by(["customer_id", "created_date"])
        .agg(pl.count("item_id").alias("num_items"))
    )

    X8 = (
        orders
        .group_by("customer_id")
        .agg(pl.col("num_items").mean().alias("X_8"))
        .collect()
    )

    # -------------------------------
    # X9: số brand khác nhau từng mua
    # -------------------------------
    X9 = (
    train_history
    .join(item_df.lazy().select(["item_id", "brand"]), on="item_id", how="left")
    .select(["customer_id", "brand"])
    .unique()
    .group_by("customer_id")
    .agg(pl.count("brand").alias("X_9"))
    .collect()
    )

    return X7, X8, X9


In [32]:
X7, X8, X9 = compute_X7_X8_X9(train_history,item_df)
X7.write_parquet("feature_X7_train.parquet", compression="snappy")
X8.write_parquet("feature_X8_train.parquet", compression="snappy")
X9.write_parquet("feature_X9_train.parquet", compression="snappy")
print("finish")

finish


In [33]:
X7, X8, X9 = compute_X7_X8_X9(train_recent,item_df)
X7.write_parquet("feature_X7_val.parquet", compression="snappy")
X8.write_parquet("feature_X8_val.parquet", compression="snappy")
X9.write_parquet("feature_X9_val.parquet", compression="snappy")
print("finish")

finish


In [34]:
def compute_X10_X11_X12(train_history, user_df, end_hist):
    """
    Tạo các feature thời gian:
    X_10: số ngày từ end_hist đến ngày tạo account
    X_11: số ngày từ lần mua cuối cùng đến end_hist
    X_12: tổng số ngày khách hàng có mua hàng (distinct purchase days)
    """
    user_df_lazy = user_df.lazy()
    train_history_lazy = train_history.lazy()

    # -------------------------------
    # X_10: số ngày từ ngày tạo account đến end_hist
    # -------------------------------
    user_df_lazy = user_df_lazy.with_columns(
        (pl.lit(end_hist) - pl.col("created_date"))
        .dt.total_days()
        .alias("X_10")
    )

    # -------------------------------
    # Ngày mua đầu tiên & cuối cùng
    # -------------------------------
    purchase_base = (
        train_history_lazy
        .filter(
            (pl.col("event_type") == "Purchase") &
            (pl.col("is_deleted") == False)
        )
    )

    purchase_dates = (
        purchase_base
        .group_by("customer_id")
        .agg([
            pl.col("created_date").min().alias("first_purchase_date"),
            pl.col("created_date").max().alias("last_purchase_date")
        ])
    )

    # -------------------------------
    # X_12: tổng số ngày có mua hàng
    # -------------------------------
    purchase_days = (
        purchase_base
        .with_columns(
            pl.col("created_date").dt.date().alias("purchase_day")
        )
        .group_by("customer_id")
        .agg(
            pl.col("purchase_day").n_unique().alias("X_12")
        )
    )

    # -------------------------------
    # Join & tạo X_11
    # -------------------------------
    features = (
        user_df_lazy
        .join(purchase_dates, on="customer_id", how="left")
        .join(purchase_days, on="customer_id", how="left")
        .with_columns(
            (pl.lit(end_hist) - pl.col("last_purchase_date"))
            .dt.total_days()
            .alias("X_11")
        )
    )

    X10 = features.select(["customer_id", "X_10"])
    X11 = features.select(["customer_id", "X_11"])
    X12 = features.select(["customer_id", "X_12"])

    return X10, X11, X12


In [35]:
X10,X11,X12=compute_X10_X11_X12(train_history, user_df, end_hist)
X10.collect().write_parquet("feature_X10_train.parquet", compression="snappy")
X11.collect().write_parquet("feature_X11_train.parquet", compression="snappy")
X12.collect().write_parquet("feature_X12_train.parquet", compression="snappy")

In [36]:
X10,X11,X12=compute_X10_X11_X12(train_recent, user_df, end_recent)
X10.collect().write_parquet("feature_X10_val.parquet", compression="snappy")
X11.collect().write_parquet("feature_X11_val.parquet", compression="snappy")
X12.collect().write_parquet("feature_X12_val.parquet", compression="snappy")

In [37]:
def compute_X13(train_history, item):
    """
    Tính feature X_13: số loại sản phẩm (category_l1) trung bình cho một lần mua hàng.
    Trả về DataFrame chỉ gồm customer_id và X_13 (giữ giá trị trung bình liên tục).
    """
    # 1️⃣ Join để lấy category_l1
    df = train_history.lazy().join(
        item.lazy().select(["item_id", "category_l1"]),
        on="item_id",
        how="left"
    )

    # 2️⃣ Số loại sản phẩm unique mỗi giao dịch
    unique_cate_per_order = (
        df.group_by(["customer_id", "date_key"])
          .agg(pl.col("category_l1").n_unique().alias("so_loai_san_pham"))
    )

    # 3️⃣ Trung bình số loại sản phẩm mỗi khách hàng
    avg_cate_customer = (
        unique_cate_per_order.group_by("customer_id")
        .agg(pl.col("so_loai_san_pham").mean().alias("X_13"))
    ).collect()

    return avg_cate_customer.lazy()

In [38]:
feature_X13 = compute_X13(train_history, item_df)
feature_X13.collect().write_parquet("feature_X13_train.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [39]:
feature_X13 = compute_X13(train_recent, item_df)
feature_X13.collect().write_parquet("feature_X13_val.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [40]:
def compute_X14(train_history, item_chunk_df, n_clusters=5):
    """
    Tính feature X_14: đặc trưng khách hàng dựa trên sản phẩm category 'Sữa'.
    Phân thành n_clusters theo giá và chọn cluster mua nhiều nhất.
    """
    # 1️⃣ Lọc sản phẩm sữa
    milk_df = (
        item_chunk_df.lazy()
        .filter(pl.col("category_l1").str.contains("Sữa", literal=False))
        .select(["item_id", "price"])
        .collect()
    )

    # 2️⃣ KMeans n_clusters cluster giá
    X = milk_df["price"].to_numpy().reshape(-1, 1)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto").fit(X)
    milk_df = milk_df.with_columns(pl.Series("cluster", kmeans.labels_))

    # 3️⃣ Xác định thứ tự cluster theo giá tăng dần → số 0..n_clusters-1
    cluster_order = np.argsort([X[kmeans.labels_ == i].mean() for i in range(n_clusters)])
    tier_map = {cluster_order[i]: i for i in range(n_clusters)}

    milk_df = milk_df.with_columns(
        pl.col("cluster").cast(pl.Int64).replace_strict(tier_map).alias("milk_price_tier_num")
    ).select(["item_id", "milk_price_tier_num"])

    # 4️⃣ Join với train_history
    milk_purchase = train_history.lazy().join(milk_df.lazy(), on="item_id", how="inner").collect()

    # 5️⃣ Group + count theo tier
    customer_milk_stats = (
        milk_purchase.group_by(["customer_id", "milk_price_tier_num"])
        .agg(pl.count("item_id").alias("purchase_count"))
        .pivot(
            values="purchase_count",
            index="customer_id",
            on="milk_price_tier_num"  # columns
        )
        .fill_null(0)
    )

    # 6️⃣ Đổi tên cột từ int → string
    col_rename = {col: str(col) for col in customer_milk_stats.columns if isinstance(col, int)}
    customer_milk_stats = customer_milk_stats.rename(col_rename)

    # 7️⃣ Đảm bảo đủ cột từ "0" đến str(n_clusters-1)
    for tier in map(str, range(n_clusters)):
        if tier not in customer_milk_stats.columns:
            customer_milk_stats = customer_milk_stats.with_columns(
                pl.lit(0).alias(tier)
            )

    # 8️⃣ Chọn tier mua nhiều nhất → X_14 là số 0..n_clusters-1
    customer_milk_stats = customer_milk_stats.with_columns(
        pl.struct(list(map(str, range(n_clusters)))).map_elements(
            lambda r: int(max(r.items(), key=lambda x: x[1])[0]),
            return_dtype=pl.Int64
        ).alias("X_14")
    )

    return customer_milk_stats

In [41]:
feature_X14 = compute_X14(train_history, item_df)
feature_X14.write_parquet("feature_X14_train.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [42]:
feature_X14 = compute_X14(train_recent, item_df)
feature_X14.write_parquet("feature_X14_val.parquet", compression="snappy", use_pyarrow=True)
print("finish")

finish


In [43]:
def compute_x15(train_history: pl.DataFrame) -> pl.DataFrame:
    """
    Compute x_15: average time gap (days) between purchases for each customer.
    If purchase_count <= 2, x_15 = 0.
    
    Parameters
    ----------
    train_history : pl.DataFrame
        Must contain:
        - customer_id
        - event_type
        - created_date (datetime)
    
    Returns
    -------
    pl.DataFrame
        Columns:
        - customer_id
        - x_15
    """

    return (
        train_history
        # 1️⃣ Chỉ lấy giao dịch mua
        .filter(pl.col("event_type") == "Purchase")
        
        # 2️⃣ Sort theo customer + thời gian
        .sort(["customer_id", "created_date"])
        
        # 3️⃣ Tính gap giữa các lần mua (ngày)
        .with_columns(
            gap_days = (
                pl.col("created_date")
                .diff()
                .dt.total_days()
                .over("customer_id")
            )
        )
        
        # 4️⃣ Gom theo customer
        .group_by("customer_id")
        .agg(
            purchase_count = pl.len(),
            avg_gap_days_raw = pl.col("gap_days").mean()
        )
        
        # 5️⃣ Nếu mua ≤ 2 lần → gán = 0
        .with_columns(
            X_15 = pl.when(pl.col("purchase_count") <= 2)
                .then(0.0)
                .otherwise(pl.col("avg_gap_days_raw"))
        )
        
        # 6️⃣ Chỉ giữ 2 cột theo yêu cầu
        .select([
            "customer_id",
            "X_15"
        ])
    )

In [44]:
df_x15 = compute_x15(train_history)
df_x15.collect().write_parquet("feature_X15_train.parquet", compression="snappy", use_pyarrow=True)


In [45]:
df_x15 = compute_x15(train_recent)
df_x15.collect().write_parquet("feature_X15_val.parquet", compression="snappy", use_pyarrow=True)


In [46]:
def compute_x16(user_df: pl.DataFrame) -> pl.DataFrame:
    return (
        user_df
        .select([
            "customer_id",
            pl.col("membership").alias("X_16")
        ])
    )


In [47]:
df_x16 = compute_x16(user_df)
df_x16.write_parquet("feature_X16_train.parquet", compression="snappy", use_pyarrow=True)

In [48]:
df_x16 = compute_x16(user_df)
df_x16.write_parquet("feature_X16_val.parquet", compression="snappy", use_pyarrow=True)

In [50]:
def compute_candidates(train_history, train_recent):
    # ----------------------------
    # Positive samples (Y = 1)
    # ----------------------------
    positives = (
        train_recent
        .select(["customer_id", "item_id"])
        .unique()
        .with_columns(pl.lit(1.0).alias("Y"))
    )
    # ----------------------------
    # Negative samples (weighted Y)
    # ----------------------------
    negatives = (
        train_history
        .select(["customer_id", "item_id", "created_date"])
        .unique(subset=["customer_id", "item_id"])
        .join(positives.select(["customer_id", "item_id"]),
              on=["customer_id", "item_id"],
              how="anti")
        .with_columns(
            pl.when(pl.col("created_date").dt.month().is_in([7, 8,9]))
              .then(0.5)
              .when(pl.col("created_date").dt.month().is_in([4, 5, 6]))
              .then(0.3)
              .when(pl.col("created_date").dt.month().is_in([1, 2, 3]))
              .then(0.2)
              .otherwise(0.0)   # fallback (phòng hờ)
              .alias("Y")
        )
        .select(["customer_id", "item_id", "Y"])
    )

    # ----------------------------
    # Merge candidates
    # ----------------------------
    candidates = pl.concat([positives, negatives])

    return candidates


In [51]:
candidates=compute_candidates(train_history, train_recent)
candidates.collect().write_parquet("candidates_train.parquet", compression="snappy", use_pyarrow=True)

In [52]:
def compute_candidates_val(train_history, train_recent):
    """
    Lấy tất cả cặp (customer_id, item_id) xuất hiện trong train_history và train_recent.
    Không gán nhãn Y.
    """
    # Lấy cặp từ recent
    recent_keys = train_recent.select(["customer_id", "item_id"]).unique()

    # Lấy cặp từ history trừ recent (nếu muốn loại bỏ trùng với recent)
    #history_keys = train_history.select(["customer_id", "item_id"]).unique()
    
    # Kết hợp 2 tập
    #candidates = pl.concat([recent_keys, history_keys]).unique()
    candidates = recent_keys
    return candidates


In [53]:
candidates=compute_candidates_val(train_history, train_recent)
candidates.collect().write_parquet("candidates_val.parquet", compression="snappy", use_pyarrow=True)

In [54]:
print("Số dòng candidates:", candidates.select(pl.len()).collect().item())

Số dòng candidates: 19217254


In [55]:
def merge_features():
    # Scan parquet (LazyFrame)
    candidates = pl.scan_parquet("candidates_train.parquet")
    X1  = pl.scan_parquet("feature_X1_train.parquet")
    X2  = pl.scan_parquet("feature_X2_train.parquet")
    X3  = pl.scan_parquet("feature_X3_train.parquet")
    X4  = pl.scan_parquet("feature_X4_train.parquet")
    X5  = pl.scan_parquet("feature_X5_train.parquet")
    X6  = pl.scan_parquet("feature_X6_train.parquet")
    X7  = pl.scan_parquet("feature_X7_train.parquet")
    X8  = pl.scan_parquet("feature_X8_train.parquet")
    X9  = pl.scan_parquet("feature_X9_train.parquet")
    X10 = pl.scan_parquet("feature_X10_train.parquet")
    X11 = pl.scan_parquet("feature_X11_train.parquet")
    X12 = pl.scan_parquet("feature_X12_train.parquet")
    X13 = pl.scan_parquet("feature_X13_train.parquet")
    X14 = pl.scan_parquet("feature_X14_train.parquet")
    X15 = pl.scan_parquet("feature_X15_train.parquet")
    X16 = pl.scan_parquet("feature_X16_train.parquet")
    # Merge features
    features = (
        candidates
        .join(X5,  on="item_id", how="left")
        .join(X1,  on=["customer_id", "item_id"], how="left")
        .join(X2,  on=["customer_id", "item_id"], how="left")
        .join(X3,  on=["customer_id", "item_id"], how="left")
        .join(X4,  on="item_id", how="left")
        .join(X6,  on="customer_id", how="left")
        .join(X7,  on="customer_id", how="left")
        .join(X8,  on="customer_id", how="left")
        .join(X9,  on="customer_id", how="left")
        .join(X10, on="customer_id", how="left")
        .join(X11, on="customer_id", how="left")
        .join(X12, on="customer_id", how="left")
        .join(X13, on="customer_id", how="left")
        .join(X14, on="customer_id", how="left")
        .join(X15, on="customer_id", how="left")
        .join(X16, on="customer_id", how="left")
        .fill_null(0)
        .select([
            "customer_id",
            "item_id",
            "X_1", "X_2", "X_3", "X_4", "X_5",
            "X_6", "X_7", "X_8", "X_9", "X_10",
            "X_11", "X_12", "X_13", "X_14", "X_15", "X_16",
            "Y"
        ])
    )
    # Ghi ra parquet
    features.collect().write_parquet(
        "features_train.parquet",
        compression="snappy",
        use_pyarrow=True
    )
    return features
features = merge_features()

In [56]:
# -----------------------------
# 3️⃣ Merge all features + candidates
# -----------------------------
def merge_features():
    candidates = pl.scan_parquet("candidates_val.parquet")
    X1 = pl.scan_parquet("feature_X1_val.parquet")
    X2 = pl.scan_parquet("feature_X2_val.parquet")
    X3 = pl.scan_parquet("feature_X3_val.parquet")
    X4 = pl.scan_parquet("feature_X4_val.parquet")
    X5 = pl.scan_parquet("feature_X5_val.parquet")
    X6 = pl.scan_parquet("feature_X6_val.parquet")
    X7 = pl.scan_parquet("feature_X7_val.parquet")
    X8 = pl.scan_parquet("feature_X8_val.parquet")
    X9 = pl.scan_parquet("feature_X9_val.parquet")
    X10 = pl.scan_parquet("feature_X10_val.parquet")
    X11 = pl.scan_parquet("feature_X11_val.parquet")
    X12 = pl.scan_parquet("feature_X12_val.parquet")
    X13 = pl.scan_parquet("feature_X13_val.parquet")
    X14 = pl.scan_parquet("feature_X14_val.parquet")
    X15 = pl.scan_parquet("feature_X15_val.parquet")
    X16 = pl.scan_parquet("feature_X16_val.parquet")

    features = (
        candidates
        .join(X5, on="item_id", how="left")
        .join(X1, on=["customer_id", "item_id"], how="left")
        .join(X2, on=["customer_id", "item_id"], how="left")
        .join(X3, on=["customer_id", "item_id"], how="left")
        .join(X4, on="item_id", how="left")
        .join(X6, on="customer_id", how="left") 
        .join(X7, on="customer_id", how="left")
        .join(X8, on="customer_id", how="left")
        .join(X9, on="customer_id", how="left")
        .join(X10, on="customer_id", how="left")
        .join(X11, on="customer_id", how="left")
        .join(X12, on="customer_id", how="left")
        .join(X13, on="customer_id", how="left")
        .join(X14, on="customer_id", how="left")
        .join(X15, on="customer_id", how="left")
        .join(X16, on="customer_id", how="left")  
        .fill_null(0)
        .select(["customer_id", "item_id", "X_1", "X_2", "X_3","X_4","X_5","X_6","X_7","X_8","X_9","X_10","X_11","X_12","X_13","X_14","X_15","X_16"])
    )

    features.collect().write_parquet("features_val.parquet", compression="snappy", use_pyarrow=True)
    return features

features = merge_features()

In [2]:
train_path = "features_train.parquet"
val_path = "features_val.parquet"

# Đọc file parquet bằng Polars
df_train = pl.read_parquet(train_path)
df_val = pl.read_parquet(val_path)

# In số dòng
print("Số dòng train:", df_train.height)
print("Số dòng validation:", df_val.height)

# Đọc 10 dòng đầu
print("\n===== 10 dòng đầu của TRAIN =====")
df_train.head(1)



Số dòng train: 26790113
Số dòng validation: 19217254

===== 10 dòng đầu của TRAIN =====


customer_id,item_id,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,X_11,X_12,X_13,X_14,X_15,X_16,Y
i32,str,u32,u32,u32,u32,u8,f64,u32,f64,u32,i64,i64,u32,f64,i64,f64,i64,f64
841313,"""4600000000001""",6,6,6,25843,19,12.8093,12,1.2,2,2878,12,6,1.833333,3,7.818182,2,1.0


In [3]:
print("\n===== 10 dòng đầu của VALIDATION =====")
df_val.head(1)


===== 10 dòng đầu của VALIDATION =====


customer_id,item_id,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,X_11,X_12,X_13,X_14,X_15,X_16
i32,str,u32,u32,u32,u32,u8,f64,u32,f64,u32,i64,i64,u32,f64,i64,f64,i64
5316154,"""0203000000002""",1,1,1,125013,0,55.061814,70,2.692308,5,959,6,13,3.307692,0,3.246377,1


In [4]:
df_y1 = df_train.filter(pl.col("Y") == 1)
df_y0 = df_train.filter(pl.col("Y") != 1)

print("Số mẫu Y=1:", df_y1.height)
print("Số mẫu Y!=1:", df_y0.height)

# Lấy ngẫu nhiên 3 triệu mẫu Y=0
df_y0_sample = df_y0.sample(n=1_000_000, shuffle=True)

# Gộp lại thành tập train balanced
df_train_balanced = pl.concat([df_y1, df_y0_sample], how="vertical")

print("Tổng số mẫu balanced:", df_train_balanced.height)

 #Nếu muốn shuffle lại toàn bộ:
df_train_balanced = df_train_balanced.sample(n=df_train_balanced.height, shuffle=True)


Số mẫu Y=1: 19217254
Số mẫu Y!=1: 7572859
Tổng số mẫu balanced: 20217254


In [5]:

# 3️⃣ Tạo lại X_test và id_cols
X_test = df_val.drop(["customer_id", "item_id"])
id_cols = df_val.select(["customer_id", "item_id"])


X_test.head(1)


X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,X_11,X_12,X_13,X_14,X_15,X_16
u32,u32,u32,u32,u8,f64,u32,f64,u32,i64,i64,u32,f64,i64,f64,i64
1,1,1,125013,0,55.061814,70,2.692308,5,959,6,13,3.307692,0,3.246377,1


In [6]:
# ======================
# 0) Các cột định danh
# ======================
key_cols = ["customer_id", "item_id"]
# ====================================================
# 5) Tạo X_train, y_train, X_test, y_test
# ====================================================
X_train = df_train.drop(["customer_id", "item_id", "Y"])
y_train = df_train["Y"]


#y_test = df_val["Y"]
id_cols = df_val.select(["customer_id", "item_id"])
# ====================================================
# 6) Debug xem sample
# ====================================================
X_train.head(1)

X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,X_11,X_12,X_13,X_14,X_15,X_16
u32,u32,u32,u32,u8,f64,u32,f64,u32,i64,i64,u32,f64,i64,f64,i64
6,6,6,25843,19,12.8093,12,1.2,2,2878,12,6,1.833333,3,7.818182,2


In [7]:
y_train.value_counts()

Y,count
f64,u32
0.2,5786325
0.3,1786534
1.0,19217254


In [8]:
X_test.head(1)

X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,X_11,X_12,X_13,X_14,X_15,X_16
u32,u32,u32,u32,u8,f64,u32,f64,u32,i64,i64,u32,f64,i64,f64,i64
1,1,1,125013,0,55.061814,70,2.692308,5,959,6,13,3.307692,0,3.246377,1


In [9]:
import polars as pl
import xgboost as xgb
import numpy as np

# ======================
# CONFIG
# ======================
feature_cols = [f"X_{i}" for i in range(1, 17)]

params = {
    "max_depth": 6,
    "learning_rate": 0.001,
    "min_child_weight": 4,
    "gamma": 0.2,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "device": "cuda",
    "max_bin": 128,
    "seed": 42
}

# ======================
# POLARS → NUMPY (NO PANDAS)
# ======================
X_train_np = (
    X_train
    .select(feature_cols)
    .with_columns(pl.all().cast(pl.Float32))
    .to_numpy(order="c")
)

y_train_np = (
    y_train
    .cast(pl.Float32)
    .to_numpy()
)

# ======================
# GPU-OPTIMIZED DMatrix
# ======================
dtrain = xgb.QuantileDMatrix(
    X_train_np,
    label=y_train_np,
    max_bin=128
)

# ======================
# TRAIN
# ======================
model = xgb.train(
    params,
    dtrain,
    num_boost_round=600
)

# ======================
# SAVE
# ======================
model.save_model("xgb_model.json")

print("✅ Train done (Polars → NumPy → QuantileDMatrix, NO pandas)")


"""from xgboost import XGBRegressor

model = XGBRegressor(
    booster="gblinear",          # 🔥 LINEAR REGRESSION
    objective="reg:squarederror",
    eval_metric="rmse",

    # Linear model params
    learning_rate=0.03,
    reg_alpha=0.1,               # L1 (Lasso)
    reg_lambda=1.0,              # L2 (Ridge)

    # GPU
    tree_method="gpu_hist",      # vẫn cần để dùng GPU
    predictor="gpu_predictor",

    random_state=42
)

# Ép float32 như cũ (OK)
X_train_np = X_train_np.astype("float32")
y_train_np = y_train_np.astype("float32")

model.fit(X_train_np, y_train_np)
"""




✅ Train done (Polars → NumPy → QuantileDMatrix, NO pandas)


'from xgboost import XGBRegressor\n\nmodel = XGBRegressor(\n    booster="gblinear",          # 🔥 LINEAR REGRESSION\n    objective="reg:squarederror",\n    eval_metric="rmse",\n\n    # Linear model params\n    learning_rate=0.03,\n    reg_alpha=0.1,               # L1 (Lasso)\n    reg_lambda=1.0,              # L2 (Ridge)\n\n    # GPU\n    tree_method="gpu_hist",      # vẫn cần để dùng GPU\n    predictor="gpu_predictor",\n\n    random_state=42\n)\n\n# Ép float32 như cũ (OK)\nX_train_np = X_train_np.astype("float32")\ny_train_np = y_train_np.astype("float32")\n\nmodel.fit(X_train_np, y_train_np)\n'

In [12]:
import polars as pl

# lấy importance (weight / gain / cover)
importance_dict = model.get_score(importance_type="gain")
feature_names = [f"X_{i}" for i in range(1, 17)]

rows = []
for i, name in enumerate(feature_names):
    rows.append({
        "feature": name,
        "importance": importance_dict.get(f"f{i}", 0.0)
    })
df_importance = (
    pl.DataFrame(rows)
    .sort("importance", descending=True)
)

df_importance.head(10)


feature,importance
str,f64
"""X_11""",34821.320312
"""X_1""",19021.853516
"""X_2""",18570.044922
"""X_3""",16171.330078
"""X_10""",13746.035156
"""X_15""",6063.500977
"""X_12""",3114.571533
"""X_16""",3095.648438
"""X_5""",1409.376709


In [13]:
import xgboost as xgb

model = xgb.Booster()
model.load_model("xgb_model.json")
model.set_param({"device": "cuda"})   # hoặc "cpu"

import numpy as np

def predict_in_batches(model, X_np, batch_size=1_000_000):
    preds = []

    for i in range(0, X_np.shape[0], batch_size):
        batch = X_np[i:i+batch_size]
        y_pred = model.inplace_predict(batch)
        preds.append(y_pred)

    return np.concatenate(preds)
    
# Polars → NumPy (NO pandas)
X_all_np = (
    X_test
    .with_columns(pl.all().cast(pl.Float32))
    .to_numpy(order="c")
)

pred_score = predict_in_batches(model, X_all_np)
'''import joblib
import numpy as np
import polars as pl

# ======================
# LOAD LINEAR MODEL (sklearn)
# ======================
model = joblib.load("linear_sgd_225M.joblib")

# ======================
# PREDICT IN BATCHES
# ======================
def predict_in_batches(model, X_np, batch_size=1_000_000):
    preds = []

    for i in range(0, X_np.shape[0], batch_size):
        batch = X_np[i:i + batch_size]
        y_pred = model.predict(batch)   # ✅ đúng cho linear regression
        preds.append(y_pred)

    return np.concatenate(preds)

# ======================
# POLARS → NUMPY
# ======================
X_all_np = (
    X_test
    .with_columns(pl.all().cast(pl.Float32))
    .to_numpy(order="c")
)

# ======================
# PREDICT
# ======================
pred_score = predict_in_batches(model, X_all_np)





In [14]:
import polars as pl
import json

# =================================================
# 1️⃣ GỘP KẾT QUẢ PREDICT (TRAIN + TEST)
# =================================================

df_test_pred = (
    X_test
    .with_columns(
        pl.Series("pred_proba", pred_score) # pred_score = predict(X_test)
    )
    .with_columns([
        df_val["customer_id"],
        df_val["item_id"]
    ])
)

#df_train_pred = (
#    X_train
#    .with_columns(
#        pl.Series("pred_proba", pred_train)   # pred_train = predict(X_train)
#    )
#   .with_columns([
#        df_train_balanced["customer_id"],
#        df_train_balanced["item_id"]
#    ])
#)

# 🔥 Gộp TRAIN + TEST
df_all = df_test_pred

# =================================================
# 2️⃣ GIỮ ITEM ĐƯỢC DỰ ĐOÁN MUA
# (hiện tại giữ tất cả)
# =================================================

df_buy = df_all

# =================================================
# 3️⃣ SORT: customer_id ↑ , pred_proba ↓
# =================================================

df_buy = df_buy.sort(
    ["customer_id", "pred_proba"],
    descending=[False, True]
)

# =================================================
# 4️⃣ MỖI CUSTOMER LẤY TỐI ĐA 10 ITEM
# =================================================

df_top10 = (
    df_buy
    .group_by("customer_id")
    .head(10)
)

# =================================================
# 5️⃣ GOM item_id → LIST
# =================================================

df_result = (
    df_top10
    .group_by("customer_id")
    .agg(
        pl.col("item_id").implode().alias("items")
    )
)

# =================================================
# 6️⃣ GHI FILE submission.json (THEO BATCH – NHẸ RAM)
# =================================================

BATCH_SIZE = 50_000   # giảm xuống 20_000 nếu RAM yếu

with open("submission.json", "w", encoding="utf-8") as f:
    f.write("{\n")

    total = df_result.height

    for start in range(0, total, BATCH_SIZE):
        batch = df_result.slice(start, BATCH_SIZE)

        for idx, row in enumerate(batch.iter_rows(named=True)):
            customer_id = str(row["customer_id"])
            items = row["items"]

            f.write(f'"{customer_id}": ')
            json.dump(items, f)

            # chỉ ghi dấu phẩy nếu CHƯA phải phần tử cuối
            if start + idx < total - 1:
                f.write(",\n")

    f.write("\n}")

print("✅ Đã xuất submission.json – gồm TRAIN + TEST, mỗi customer tối đa 10 item")


✅ Đã xuất submission.json – gồm TRAIN + TEST, mỗi customer tối đa 10 item


In [11]:
import pickle
import json
import numpy as np

def precision_at_k(pred, gt, hist=None, filter_bought_items=True, K=10):
    """
    pred: dict {customer_id: [item_id1, item_id2, ...]} - file submission
    gt: dict {customer_id: [item_id1, item_id2, ...]} - file groundtruth
    hist: dict {customer_id: [item_id1, ...]} - lịch sử mua của user, optional
    filter_bought_items: nếu True, loại bỏ các item đã mua trong history
    K: đánh giá top-K
    """
    precisions = []
    cold_start_users = []

    for user in gt.keys():
        if user not in pred:
            cold_start_users.append(user)
            continue

        gt_items = set(gt[user])
        if filter_bought_items and hist is not None and user in hist:
            gt_items -= set(hist[user])

        hits = len(set(pred[user][:K]) & gt_items)
        precisions.append(hits / K)

    mean_precision = np.mean(precisions) if precisions else 0.0
    return mean_precision, cold_start_users

# --- Load groundtruth ---
with open("final_groundtruth_clean.pkl", "rb") as f:
    groundtruth = pickle.load(f)

print(f"Số customer trong groundtruth: {len(groundtruth)}")

# --- Load submission ---
with open("submission_with_hist.json", "r") as f:
    submission = json.load(f)

# Nếu keys trong submission là str mà groundtruth là int, chuyển sang int
try:
    submission_fixed = {int(k): v for k, v in submission.items()}
except ValueError:
    submission_fixed = submission  # giữ nguyên nếu không thể chuyển

print(f"Số customer trong submission: {len(submission_fixed)}")
 
# --- Tính precision@K ---
precision, cold_users = precision_at_k(submission_fixed, groundtruth, hist=None, K=10)
print(f"Precision@10: {precision:.6f}")
print(f"Số cold start users (không có dự đoán): {len(cold_users)}")


Số customer trong groundtruth: 644970
Số customer trong submission: 2094582
Precision@10: 0.093271
Số cold start users (không có dự đoán): 148385


In [31]:
import pickle
import json
import numpy as np
import pandas as pd

# ================= FUNCTION =================
def precision_at_k(pred, gt, hist=None, filter_bought_items=True, K=10, fill_cold_start=None):
    precisions = []

    for user in gt.keys():
        if user in pred:
            user_pred = pred[user][:K]
        else:
            if fill_cold_start is not None:
                user_pred = fill_cold_start[:K]
            else:
                user_pred = []

        gt_items = set(gt[user])
        if filter_bought_items and hist is not None and user in hist:
            gt_items -= set(hist[user])

        hits = len(set(user_pred) & gt_items)
        precisions.append(hits / K)

    return np.mean(precisions) if precisions else 0.0


# ================= LOAD GROUNDTRUTH =================
with open("final_groundtruth_clean.pkl", "rb") as f:
    groundtruth = pickle.load(f)

print(f"👥 Số customer trong groundtruth: {len(groundtruth)}")


# ================= LOAD SUBMISSION =================
with open("submission_with_hist.json", "r", encoding="utf-8") as f:
    submission = json.load(f)

# fix key type nếu cần
try:
    submission = {int(k): v for k, v in submission.items()}
except ValueError:
    pass

print(f"📁 Số customer trong submission gốc: {len(submission)}")


# ================= LOAD TOP ITEMS =================
top_items_df = pd.read_csv("top10_first_purchase_by_count.csv")
top_items = top_items_df["item_id"].tolist()


# ================= FILL COLD START (KHÔNG LOẠI CUSTOMER) =================

# 1️⃣ Giữ nguyên toàn bộ submission gốc
submission_filled = {
    user: items[:10]
    for user, items in submission.items()
}

# 2️⃣ Chỉ fill customer thiếu so với groundtruth
cold_start_count = 0
for user in groundtruth.keys():
    if user not in submission_filled:
        submission_filled[user] = top_items[:10]
        cold_start_count += 1

print(f"❄️ Cold-start customers được fill: {cold_start_count}")
print(f"✅ Tổng customer sau fill: {len(submission_filled)}")


# ================= SAVE FILLED SUBMISSION =================
with open("submission_filled.json", "w", encoding="utf-8") as f:
    json.dump(submission_filled, f, ensure_ascii=False)

print("💾 Đã lưu file submission_filled.json")


# ================= EVALUATE =================
precision = precision_at_k(
    submission_filled,
    groundtruth,
    hist=None,
    K=10,
    fill_cold_start=None   # đã fill rồi nên không cần nữa
)

print(f"🎯 Precision@10 (sau khi fill & lưu): {precision:.6f}")



👥 Số customer trong groundtruth: 644970
📁 Số customer trong submission gốc: 2107732
❄️ Cold-start customers được fill: 135235
✅ Tổng customer sau fill: 2242967
💾 Đã lưu file submission_filled.json
🎯 Precision@10 (sau khi fill & lưu): 0.080587


In [28]:
import polars as pl
import math

# Load dữ liệu
customers = pl.read_parquet("customer_id_predict.parquet").select("customer_id")
items = pl.read_parquet("final_candidate_items.parquet").select("item_id")

n_items = items.height
target_rows_per_chunk = 1_000_000

# Số customer mỗi chunk ≈ 1M / n_items
customers_per_chunk = max(1, target_rows_per_chunk // n_items)

print("Items:", n_items)
print("Customers per chunk:", customers_per_chunk)

total_customers = customers.height
chunk_idx = 0
total_rows = 0

for start in range(0, total_customers, customers_per_chunk):
    cust_chunk = customers.slice(start, customers_per_chunk)

    joined = cust_chunk.join(items, how="cross")
    rows = joined.height
    total_rows += rows

    out_file = f"customer_item_keys_chunk_{chunk_idx}.parquet"
    joined.write_parquet(out_file)

    print(f"Saved {out_file} with {rows:,} rows")

    chunk_idx += 1

print(f"Total rows generated: {total_rows:,}")

Items: 97
Customers per chunk: 10309
Saved customer_item_keys_chunk_0.parquet with 999,973 rows
Saved customer_item_keys_chunk_1.parquet with 999,973 rows
Saved customer_item_keys_chunk_2.parquet with 999,973 rows
Saved customer_item_keys_chunk_3.parquet with 999,973 rows
Saved customer_item_keys_chunk_4.parquet with 999,973 rows
Saved customer_item_keys_chunk_5.parquet with 999,973 rows
Saved customer_item_keys_chunk_6.parquet with 999,973 rows
Saved customer_item_keys_chunk_7.parquet with 999,973 rows
Saved customer_item_keys_chunk_8.parquet with 999,973 rows
Saved customer_item_keys_chunk_9.parquet with 999,973 rows
Saved customer_item_keys_chunk_10.parquet with 999,973 rows
Saved customer_item_keys_chunk_11.parquet with 999,973 rows
Saved customer_item_keys_chunk_12.parquet with 999,973 rows
Saved customer_item_keys_chunk_13.parquet with 999,973 rows
Saved customer_item_keys_chunk_14.parquet with 999,973 rows
Saved customer_item_keys_chunk_15.parquet with 999,973 rows
Saved custome

In [86]:
import polars as pl
from pathlib import Path

CHUNK_FOLDER = "cus_item_key"
PURCHASED_FILE = "item_Da_mua/customer_item_purchased.parquet"
OUTPUT_FOLDER = "cus_item_key_filter"

Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

# Load các cặp đã mua (lazy, streaming)
purchased = pl.scan_parquet(PURCHASED_FILE).select(["customer_id", "item_id"])

chunk_files = sorted(Path(CHUNK_FOLDER).glob("*.parquet"))
print(f"🔹 Found {len(chunk_files)} chunks")

for i, chunk_path in enumerate(chunk_files, 1):
    print(f"[{i}/{len(chunk_files)}] Filtering {chunk_path.name}")

    chunk = pl.scan_parquet(chunk_path)

    # left join anti: giữ các cặp không có trong purchased
    filtered = chunk.join(
        purchased,
        on=["customer_id", "item_id"],
        how="anti"
    )

    # ghi ra output
    out_path = Path(OUTPUT_FOLDER) / chunk_path.name
    filtered.collect(streaming=True).write_parquet(out_path)

    print(f"  ✅ Saved filtered chunk → {out_path.name}")

print("🎉 DONE: All chunks filtered")


🔹 Found 204 chunks
[1/204] Filtering customer_item_keys_chunk_0.parquet


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_3540\2927891684.py:30: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  filtered.collect(streaming=True).write_parquet(out_path)


  ✅ Saved filtered chunk → customer_item_keys_chunk_0.parquet
[2/204] Filtering customer_item_keys_chunk_1.parquet
  ✅ Saved filtered chunk → customer_item_keys_chunk_1.parquet
[3/204] Filtering customer_item_keys_chunk_10.parquet
  ✅ Saved filtered chunk → customer_item_keys_chunk_10.parquet
[4/204] Filtering customer_item_keys_chunk_100.parquet
  ✅ Saved filtered chunk → customer_item_keys_chunk_100.parquet
[5/204] Filtering customer_item_keys_chunk_101.parquet
  ✅ Saved filtered chunk → customer_item_keys_chunk_101.parquet
[6/204] Filtering customer_item_keys_chunk_102.parquet
  ✅ Saved filtered chunk → customer_item_keys_chunk_102.parquet
[7/204] Filtering customer_item_keys_chunk_103.parquet
  ✅ Saved filtered chunk → customer_item_keys_chunk_103.parquet
[8/204] Filtering customer_item_keys_chunk_104.parquet
  ✅ Saved filtered chunk → customer_item_keys_chunk_104.parquet
[9/204] Filtering customer_item_keys_chunk_105.parquet
  ✅ Saved filtered chunk → customer_item_keys_chunk_105.

In [24]:
import polars as pl
from pathlib import Path

# ========= CONFIG =========
VAL_FILE = "features_train.parquet"

OUTPUT_DIR = Path("features_split")
OUTPUT_DIR.mkdir(exist_ok=True)

# ========= LOAD =========
features = pl.read_parquet(VAL_FILE)

# ========= DROP Y IF EXISTS =========
if "Y" in features.columns:
    features = features.drop("Y")

print("Loaded VAL (Y removed if existed)")

# ========= 1. customer-item =========
feat_cus_item = (
    features
    .select([
        "customer_id",
        "item_id",
        "X_1", "X_2", "X_3"
    ])
    .unique(
        subset=["customer_id", "item_id"],
        keep="first"
    )
)

feat_cus_item.write_parquet(OUTPUT_DIR / "feat_cus_item.parquet")
print("Saved feat_cus_item.parquet")

# ========= 2. item =========
feat_item = (
    features
    .select([
        "item_id",
        "X_4", "X_5"
    ])
    .unique(
        subset=["item_id"],
        keep="first"
    )
)

feat_item.write_parquet(OUTPUT_DIR / "feat_item.parquet")
print("Saved feat_item.parquet")

# ========= 3. customer =========
feat_cus = (
    features
    .select([
        "customer_id",
        "X_6", "X_7", "X_8",
        "X_9", "X_10", "X_11",
        "X_12", "X_13", "X_14",
        "X_15", "X_16"
    ])
    .unique(
        subset=["customer_id"],
        keep="first"
    )
)

feat_cus.write_parquet(OUTPUT_DIR / "feat_cus.parquet")
print("Saved feat_cus.parquet")

print("🎉 DONE: features_split created from VAL only")



Loaded VAL (Y removed if existed)
Saved feat_cus_item.parquet
Saved feat_item.parquet
Saved feat_cus.parquet
🎉 DONE: features_split created from VAL only


In [25]:
pl.scan_parquet("features_split/feat_cus_item.parquet") \
  .unique(["customer_id", "item_id"]) \
  .collect(engine="streaming") \
  .write_parquet("features_split/feat_cus_item_u.parquet")

pl.scan_parquet("features_split/feat_item.parquet") \
  .unique("item_id") \
  .collect(engine="streaming") \
  .write_parquet("features_split/feat_item_u.parquet")

pl.scan_parquet("features_split/feat_cus.parquet") \
  .unique("customer_id") \
  .collect(engine="streaming") \
  .write_parquet("features_split/feat_cus_u.parquet")

print("finish")

finish


In [26]:
import polars as pl
from pathlib import Path

FEATURE_COLS = [f"X_{i}" for i in range(1, 17)]

def join_features_chunks_pipeline_merge_all_add_Y(
    test_folder: str,
    feat_cus_item_parquet: str,
    feat_item_parquet: str,
    feat_cus_parquet: str,
    output_folder: str
):
    test_folder = Path(test_folder)
    out_dir = Path(output_folder)
    out_dir.mkdir(exist_ok=True)

    # Load features (lazy)
    feat_cus_item = pl.scan_parquet(feat_cus_item_parquet)
    feat_item     = pl.scan_parquet(feat_item_parquet)
    feat_cus      = pl.scan_parquet(feat_cus_parquet)

    test_files = sorted(test_folder.glob("*.parquet"))
    print(f"🔹 Found {len(test_files)} test chunks")

    for idx, test_path in enumerate(test_files, 1):
        print(f"\n▶ [{idx}/{len(test_files)}] {test_path.name}")

        test = pl.scan_parquet(test_path)

        joined = (
            test
            .join(feat_item, on="item_id", how="left")
            .join(feat_cus, on="customer_id", how="left")
            .join(feat_cus_item, on=["customer_id", "item_id"], how="left")
            .fill_null(0)
            # 👉 thêm cột Y = 0.2
            .with_columns(
                pl.lit(0.2).alias("Y")
            )
        )

        # sắp xếp lại cột
        final_cols = (
            ["customer_id", "item_id"]
            + FEATURE_COLS
            + ["Y"]
        )

        joined = joined.select(final_cols)

        joined.collect(streaming=True).write_parquet(out_dir / test_path.name)
        print("  ✅ saved MERGED + X_1..X_16 + Y=0.2")

    print("\n🎉 DONE: added Y = 0.2 for all rows")



In [27]:
join_features_chunks_pipeline_merge_all_add_Y(
    test_folder="cus_item_key_filter",          # folder chứa các chunk test
    feat_cus_item_parquet="features_split/feat_cus_item_u.parquet",
    feat_item_parquet="features_split/feat_item_u.parquet",
    feat_cus_parquet="features_split/feat_cus_u.parquet",
    output_folder="cus_item_key_features_train"          # output merged + Y
)


🔹 Found 204 test chunks

▶ [1/204] customer_item_keys_chunk_0.parquet


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17996\3580755331.py:51: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  joined.collect(streaming=True).write_parquet(out_dir / test_path.name)


  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [2/204] customer_item_keys_chunk_1.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [3/204] customer_item_keys_chunk_10.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [4/204] customer_item_keys_chunk_100.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [5/204] customer_item_keys_chunk_101.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [6/204] customer_item_keys_chunk_102.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [7/204] customer_item_keys_chunk_103.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [8/204] customer_item_keys_chunk_104.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [9/204] customer_item_keys_chunk_105.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [10/204] customer_item_keys_chunk_106.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [11/204] customer_item_keys_chunk_107.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [12/204] customer_item_keys_chunk_108.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [13/204] customer_item_ke

In [28]:
df= pl.read_parquet("cus_item_key_features_train/customer_item_keys_chunk_95.parquet")
df.head(1)

customer_id,item_id,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,X_11,X_12,X_13,X_14,X_15,X_16,Y
i32,str,u32,u32,u32,u32,u8,f64,u32,f64,u32,i64,i64,u32,f64,i64,f64,i64,f64
8197400,"""4759000000002""",0,0,0,4436,0,0.0,0,0.0,0,-28,0,0,0.0,0,0.0,2,0.2


In [10]:
import polars as pl
from pathlib import Path
import xgboost as xgb
import numpy as np

# ================= CONFIG =================
PARQUET_DIR = Path("cus_item_key_features_train")

FEATURE_COLS = [f"X_{i}" for i in range(1, 17)]
LABEL_COL = "Y"

BATCH_SIZE = 400_000      # chỉnh theo RAM / VRAM
ROUNDS_PER_BATCH = 10    # số cây cho mỗi batch
MODEL_PATH = "xgb_model_stream.json"

DEVICE = "cuda"   # "cuda" hoặc "cpu"
# =========================================


# ========= XGBOOST PARAMS =========
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.05,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "device": DEVICE,
    "seed": 42,
}

# ========= LOAD FILE LIST =========
files = sorted(PARQUET_DIR.glob("customer_item_keys_chunk_*.parquet"))
print(f"🔹 Found {len(files)} parquet files")

booster = None
total_trained = 0

# ========= TRAIN STREAMING =========
for fi, pq in enumerate(files, 1):
    print(f"\n▶ [{fi}/{len(files)}] Training {pq.name}")

    lf = pl.scan_parquet(pq)
    n_rows = lf.select(pl.len()).collect().item()
    print(f"   Rows: {n_rows:,}")

    for start in range(0, n_rows, BATCH_SIZE):
        batch = (
            lf
            .slice(start, BATCH_SIZE)
            .select(FEATURE_COLS + [LABEL_COL])
            .collect(streaming=True)
        )

        X = batch.select(FEATURE_COLS).to_numpy()
        y = batch[LABEL_COL].to_numpy()

        dtrain = xgb.DMatrix(X, label=y)

        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=ROUNDS_PER_BATCH,
            xgb_model=booster,   # ⭐ tiếp tục train
            verbose_eval=False
        )

        total_trained += len(y)
        print(f"   ✓ trained {total_trained:,} rows")

print("\n🎉 DONE: trained XGBoost streaming")
print(f"📊 Total rows trained: {total_trained:,}")

# ========= SAVE MODEL =========
booster.save_model(MODEL_PATH)
print(f"💾 Model saved to: {MODEL_PATH}")



🔹 Found 231 parquet files

▶ [1/231] Training customer_item_keys_chunk_0.parquet
   Rows: 982,305


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_25560\3367675987.py:53: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


   ✓ trained 400,000 rows
   ✓ trained 800,000 rows
   ✓ trained 982,305 rows

▶ [2/231] Training customer_item_keys_chunk_1.parquet
   Rows: 981,920
   ✓ trained 1,382,305 rows
   ✓ trained 1,782,305 rows
   ✓ trained 1,964,225 rows

▶ [3/231] Training customer_item_keys_chunk_10.parquet
   Rows: 982,497
   ✓ trained 2,364,225 rows
   ✓ trained 2,764,225 rows
   ✓ trained 2,946,722 rows

▶ [4/231] Training customer_item_keys_chunk_100.parquet
   Rows: 981,667
   ✓ trained 3,346,722 rows
   ✓ trained 3,746,722 rows
   ✓ trained 3,928,389 rows

▶ [5/231] Training customer_item_keys_chunk_101.parquet
   Rows: 982,132
   ✓ trained 4,328,389 rows
   ✓ trained 4,728,389 rows
   ✓ trained 4,910,521 rows

▶ [6/231] Training customer_item_keys_chunk_102.parquet
   Rows: 982,608
   ✓ trained 5,310,521 rows
   ✓ trained 5,710,521 rows
   ✓ trained 5,893,129 rows

▶ [7/231] Training customer_item_keys_chunk_103.parquet
   Rows: 982,694
   ✓ trained 6,293,129 rows
   ✓ trained 6,693,129 rows
   ✓ 

In [2]:
import polars as pl
from pathlib import Path

# ========= CONFIG =========
VAL_FILE = "features_val.parquet"

OUTPUT_DIR = Path("features_split_test")
OUTPUT_DIR.mkdir(exist_ok=True)

# ========= LOAD =========
features = pl.read_parquet(VAL_FILE)

# ========= DROP Y IF EXISTS =========
if "Y" in features.columns:
    features = features.drop("Y")

print("Loaded VAL (Y removed if existed)")

# ========= 1. customer-item =========
feat_cus_item = (
    features
    .select([
        "customer_id",
        "item_id",
        "X_1", "X_2", "X_3"
    ])
    .unique(
        subset=["customer_id", "item_id"],
        keep="first"
    )
)

feat_cus_item.write_parquet(OUTPUT_DIR / "feat_cus_item.parquet")
print("Saved feat_cus_item.parquet")

# ========= 2. item =========
feat_item = (
    features
    .select([
        "item_id",
        "X_4", "X_5"
    ])
    .unique(
        subset=["item_id"],
        keep="first"
    )
)

feat_item.write_parquet(OUTPUT_DIR / "feat_item.parquet")
print("Saved feat_item.parquet")

# ========= 3. customer =========
feat_cus = (
    features
    .select([
        "customer_id",
        "X_6", "X_7", "X_8",
        "X_9", "X_10", "X_11",
        "X_12", "X_13", "X_14",
        "X_15", "X_16"
    ])
    .unique(
        subset=["customer_id"],
        keep="first"
    )
)

feat_cus.write_parquet(OUTPUT_DIR / "feat_cus.parquet")
print("Saved feat_cus.parquet")

print("🎉 DONE: features_split created from VAL only")



Loaded VAL (Y removed if existed)
Saved feat_cus_item.parquet
Saved feat_item.parquet
Saved feat_cus.parquet
🎉 DONE: features_split created from VAL only


In [3]:
pl.scan_parquet("features_split_test/feat_cus_item.parquet") \
  .unique(["customer_id", "item_id"]) \
  .collect(engine="streaming") \
  .write_parquet("features_split_test/feat_cus_item_u.parquet")

pl.scan_parquet("features_split_test/feat_item.parquet") \
  .unique("item_id") \
  .collect(engine="streaming") \
  .write_parquet("features_split_test/feat_item_u.parquet")

pl.scan_parquet("features_split_test/feat_cus.parquet") \
  .unique("customer_id") \
  .collect(engine="streaming") \
  .write_parquet("features_split_test/feat_cus_u.parquet")

print("finish")

finish


In [7]:
import polars as pl
from pathlib import Path

FEATURE_COLS = [f"X_{i}" for i in range(1, 17)]

def join_features_chunks_pipeline_merge_all_add_Y(
    test_folder: str,
    feat_cus_item_parquet: str,
    feat_item_parquet: str,
    feat_cus_parquet: str,
    output_folder: str
):
    test_folder = Path(test_folder)
    out_dir = Path(output_folder)
    out_dir.mkdir(exist_ok=True)

    # Load features (lazy)
    feat_cus_item = pl.scan_parquet(feat_cus_item_parquet)
    feat_item     = pl.scan_parquet(feat_item_parquet)
    feat_cus      = pl.scan_parquet(feat_cus_parquet)

    test_files = sorted(test_folder.glob("*.parquet"))
    print(f"🔹 Found {len(test_files)} test chunks")

    for idx, test_path in enumerate(test_files, 1):
        print(f"\n▶ [{idx}/{len(test_files)}] {test_path.name}")

        test = pl.scan_parquet(test_path)

        joined = (
            test
            .join(feat_item, on="item_id", how="left")
            .join(feat_cus, on="customer_id", how="left")
            .join(feat_cus_item, on=["customer_id", "item_id"], how="left")
            .fill_null(0)
            # 👉 thêm cột Y = 0.2
        )

        # sắp xếp lại cột
        final_cols = (
            ["customer_id", "item_id"]
            + FEATURE_COLS
        )

        joined = joined.select(final_cols)

        joined.collect(streaming=True).write_parquet(out_dir / test_path.name)
        print("  ✅ saved MERGED + X_1..X_16 + Y=0.2")

    print("\n🎉 DONE: added Y = 0.2 for all rows")

In [9]:
join_features_chunks_pipeline_merge_all_add_Y(
    test_folder="cus_item_key_filter",          # folder chứa các chunk test
    feat_cus_item_parquet="features_split_test/feat_cus_item_u.parquet",
    feat_item_parquet="features_split_test/feat_item_u.parquet",
    feat_cus_parquet="features_split_test/feat_cus_u.parquet",
    output_folder="cus_item_key_features_test"      )

🔹 Found 204 test chunks

▶ [1/204] customer_item_keys_chunk_0.parquet


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4260\2668680823.py:47: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  joined.collect(streaming=True).write_parquet(out_dir / test_path.name)


  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [2/204] customer_item_keys_chunk_1.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [3/204] customer_item_keys_chunk_10.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [4/204] customer_item_keys_chunk_100.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [5/204] customer_item_keys_chunk_101.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [6/204] customer_item_keys_chunk_102.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [7/204] customer_item_keys_chunk_103.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [8/204] customer_item_keys_chunk_104.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [9/204] customer_item_keys_chunk_105.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [10/204] customer_item_keys_chunk_106.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [11/204] customer_item_keys_chunk_107.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [12/204] customer_item_keys_chunk_108.parquet
  ✅ saved MERGED + X_1..X_16 + Y=0.2

▶ [13/204] customer_item_ke

In [1]:
import polars as pl
from pathlib import Path
import numpy as np
import xgboost as xgb
import gc
import re

# ======================
# 🔧 CONFIG
# ======================
DATA_FOLDER = "cus_item_key_features_test"
MODEL_PATH = "xgb_model_stream.json"      # 👈 model XGB
OUTPUT_FOLDER = "prediction_3cols_all_files"

BATCH_SIZE = 500_000

FEATURE_COLS = [f"X_{i}" for i in range(1, 17)]
START_CHUNK = 0
END_CHUNK = 231   # inclusive

DEVICE = "cuda"   # "cuda" hoặc "cpu"

Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

# ======================
# 🚀 LOAD MODEL
# ======================
model = xgb.Booster()
model.load_model(MODEL_PATH)
model.set_param({"device": DEVICE})

print("✅ XGBoost model loaded")

# ======================
# 📁 LOAD & FILTER PARQUET FILES
# ======================
all_files = sorted(Path(DATA_FOLDER).glob("customer_item_keys_chunk_*.parquet"))

def extract_chunk_id(path: Path) -> int:
    m = re.search(r"chunk_(\d+)", path.name)
    return int(m.group(1)) if m else -1

files = [
    f for f in all_files
    if START_CHUNK <= extract_chunk_id(f) <= END_CHUNK
]

print(f"\n📂 Found {len(files)} parquet files (chunk {START_CHUNK} → {END_CHUNK})")

if not files:
    raise RuntimeError("❌ No parquet files in selected chunk range!")

# ======================
# 🔮 PREDICT LOOP
# ======================
for idx, file in enumerate(files, 1):
    chunk_id = extract_chunk_id(file)
    print(f"\n[{idx}/{len(files)}] 🔮 Predicting {file.name} (chunk {chunk_id})")

    df = pl.read_parquet(file)

    if "Y" in df.columns:
        df = df.drop("Y")

    n_rows = df.height
    out_path = Path(OUTPUT_FOLDER) / file.name

    first_batch = True

    for start in range(0, n_rows, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_rows)
        batch = df.slice(start, end - start)

        X = (
            batch
            .select(FEATURE_COLS)
            .with_columns(pl.all().cast(pl.Float32))
            .to_numpy(order="c")
        )

        dtest = xgb.DMatrix(X)
        y_pred = model.predict(dtest)

        df_out = pl.DataFrame({
            "customer_id": batch["customer_id"],
            "item_id": batch["item_id"],
            "y_pred": y_pred
        })

        if first_batch:
            df_out.write_parquet(out_path)
            first_batch = False
        else:
            pl.concat(
                [pl.read_parquet(out_path), df_out],
                how="vertical"
            ).write_parquet(out_path)

        del X, dtest, y_pred, df_out, batch
        gc.collect()

    del df
    gc.collect()

print("\n🎉 DONE: Predicted chunks using XGBoost")



✅ XGBoost model loaded

📂 Found 204 parquet files (chunk 0 → 231)

[1/204] 🔮 Predicting customer_item_keys_chunk_0.parquet (chunk 0)

[2/204] 🔮 Predicting customer_item_keys_chunk_1.parquet (chunk 1)

[3/204] 🔮 Predicting customer_item_keys_chunk_10.parquet (chunk 10)

[4/204] 🔮 Predicting customer_item_keys_chunk_100.parquet (chunk 100)

[5/204] 🔮 Predicting customer_item_keys_chunk_101.parquet (chunk 101)

[6/204] 🔮 Predicting customer_item_keys_chunk_102.parquet (chunk 102)

[7/204] 🔮 Predicting customer_item_keys_chunk_103.parquet (chunk 103)

[8/204] 🔮 Predicting customer_item_keys_chunk_104.parquet (chunk 104)

[9/204] 🔮 Predicting customer_item_keys_chunk_105.parquet (chunk 105)

[10/204] 🔮 Predicting customer_item_keys_chunk_106.parquet (chunk 106)

[11/204] 🔮 Predicting customer_item_keys_chunk_107.parquet (chunk 107)

[12/204] 🔮 Predicting customer_item_keys_chunk_108.parquet (chunk 108)

[13/204] 🔮 Predicting customer_item_keys_chunk_109.parquet (chunk 109)

[14/204] 🔮 Predi

In [2]:
import polars as pl
from glob import glob
import json

# ===========================
# 1️⃣ Lấy danh sách file parquet
# ===========================
files = sorted(glob("prediction_3cols_all_files/customer_item_keys_chunk_*.parquet"))

# ===========================
# 2️⃣ Đọc và gộp tất cả các chunk
# ===========================
dfs = []
for f in files:
    df = pl.read_parquet(f)
    if df.height > 0:  # bỏ file rỗng
        dfs.append(df)

if not dfs:
    raise ValueError("Không có DataFrame nào được load!")

all_df = pl.concat(dfs)

# ===========================
# 3️⃣ Sort: customer_id ↑ , Y ↓
# ===========================
all_df = all_df.sort([
    pl.col("customer_id"), 
    pl.col("y_pred").reverse()  # Y giảm dần
])

# ===========================
# 4️⃣ Lấy top 10 item mỗi customer
# ===========================
top10_df = all_df.group_by("customer_id").head(10)

# ===========================
# 5️⃣ Gom item_id thành list
# ===========================
df_result = top10_df.group_by("customer_id").agg(
    pl.col("item_id").implode().alias("items")
)

# ===========================
# 6️⃣ Ghi JSON theo batch để tránh tràn RAM
# ===========================
BATCH_SIZE = 100_000
total = df_result.height

with open("submission2.json", "w", encoding="utf-8") as f:
    f.write("{\n")
    for start in range(0, total, BATCH_SIZE):
        batch = df_result.slice(start, BATCH_SIZE)
        for idx, row in enumerate(batch.iter_rows(named=True)):
            customer_id = str(row["customer_id"])
            items = row["items"]
            f.write(f'"{customer_id}": ')
            json.dump(items, f)
            if start + idx < total - 1:
                f.write(",\n")
    f.write("\n}")

print("✅ Đã xuất submission2.json – mỗi customer tối đa 10 item")

✅ Đã xuất submission2.json – mỗi customer tối đa 10 item


In [5]:
import pickle
import json
import numpy as np

def precision_at_k(pred, gt, hist=None, filter_bought_items=True, K=10):
    """
    pred: dict {customer_id: [item_id1, item_id2, ...]} - file submission
    gt: dict {customer_id: [item_id1, item_id2, ...]} - file groundtruth
    hist: dict {customer_id: [item_id1, ...]} - lịch sử mua của user, optional
    filter_bought_items: nếu True, loại bỏ các item đã mua trong history
    K: đánh giá top-K
    """
    precisions = []
    cold_start_users = []

    for user in gt.keys():
        if user not in pred:
            cold_start_users.append(user)
            continue

        gt_items = set(gt[user])
        if filter_bought_items and hist is not None and user in hist:
            gt_items -= set(hist[user])

        hits = len(set(pred[user][:K]) & gt_items)
        precisions.append(hits / K)

    mean_precision = np.mean(precisions) if precisions else 0.0
    return mean_precision, cold_start_users

# --- Load groundtruth ---
with open("final_groundtruth_clean.pkl", "rb") as f:
    groundtruth = pickle.load(f)

print(f"Số customer trong groundtruth: {len(groundtruth)}")

# --- Load submission ---
with open("submission/submission_without_hist.json", "r") as f:
    submission = json.load(f)

# Nếu keys trong submission là str mà groundtruth là int, chuyển sang int
try:
    submission_fixed = {int(k): v for k, v in submission.items()}
except ValueError:
    submission_fixed = submission  # giữ nguyên nếu không thể chuyển

print(f"Số customer trong submission: {len(submission_fixed)}")
 
# --- Tính precision@K ---
precision, cold_users = precision_at_k(submission_fixed, groundtruth, hist=None, K=10)
print(f"Precision@10: {precision:.6f}")
print(f"Số cold start users (không có dự đoán): {len(cold_users)}")


Số customer trong groundtruth: 644970
Số customer trong submission: 2094582
Precision@10: 0.034826
Số cold start users (không có dự đoán): 148385


In [34]:
import pickle
import json
import numpy as np
import pandas as pd

# ================= FUNCTION =================
def precision_at_k(pred, gt, hist=None, filter_bought_items=True, K=10, fill_cold_start=None):
    precisions = []

    for user in gt.keys():
        if user in pred:
            user_pred = pred[user][:K]
        else:
            if fill_cold_start is not None:
                user_pred = fill_cold_start[:K]
            else:
                user_pred = []

        gt_items = set(gt[user])
        if filter_bought_items and hist is not None and user in hist:
            gt_items -= set(hist[user])

        hits = len(set(user_pred) & gt_items)
        precisions.append(hits / K)

    return np.mean(precisions) if precisions else 0.0


# ================= LOAD GROUNDTRUTH =================
with open("final_groundtruth_clean.pkl", "rb") as f:
    groundtruth = pickle.load(f)

print(f"👥 Số customer trong groundtruth: {len(groundtruth)}")


# ================= LOAD SUBMISSION =================
with open("submission_without_hist.json", "r", encoding="utf-8") as f:
    submission = json.load(f)

# fix key type nếu cần
try:
    submission = {int(k): v for k, v in submission.items()}
except ValueError:
    pass

print(f"📁 Số customer trong submission gốc: {len(submission)}")


# ================= LOAD TOP ITEMS =================
top_items_df = pd.read_csv("top10_first_purchase_by_count.csv")
top_items = top_items_df["item_id"].tolist()


# ================= FILL COLD START (KHÔNG LOẠI CUSTOMER) =================

# 1️⃣ Giữ nguyên toàn bộ submission gốc
submission_filled = {
    user: items[:10]
    for user, items in submission.items()
}

# 2️⃣ Chỉ fill customer thiếu so với groundtruth
cold_start_count = 0
for user in groundtruth.keys():
    if user not in submission_filled:
        submission_filled[user] = top_items[:10]
        cold_start_count += 1

print(f"❄️ Cold-start customers được fill: {cold_start_count}")
print(f"✅ Tổng customer sau fill: {len(submission_filled)}")


# ================= SAVE FILLED SUBMISSION =================
with open("submission_without_hist_full.json", "w", encoding="utf-8") as f:
    json.dump(submission_filled, f, ensure_ascii=False)

print("💾 Đã lưu file submission_filled.json")


# ================= EVALUATE =================
precision = precision_at_k(
    submission_filled,
    groundtruth,
    hist=None,
    K=10,
    fill_cold_start=None   # đã fill rồi nên không cần nữa
)

print(f"🎯 Precision@10 (sau khi fill & lưu): {precision:.6f}")


👥 Số customer trong groundtruth: 644970
📁 Số customer trong submission gốc: 2094582
❄️ Cold-start customers được fill: 148385
✅ Tổng customer sau fill: 2242967
💾 Đã lưu file submission_filled.json
🎯 Precision@10 (sau khi fill & lưu): 0.026813


In [7]:
import pickle
import json
import numpy as np
import pandas as pd
def precision_at_k(pred, gt, hist=None, filter_bought_items=True, K=10):
    precisions = []
    cold_start_users = []
    per_user_result = []   # 👈 lưu kết quả từng user

    for user in gt.keys():
        if user not in pred:
            cold_start_users.append(user)
            continue

        pred_items = pred[user][:K]
        gt_items = set(gt[user])

        if filter_bought_items and hist is not None and user in hist:
            gt_items -= set(hist[user])

        hits = set(pred_items) & gt_items
        precision_u = len(hits) / K

        precisions.append(precision_u)

        per_user_result.append({
            "customer_id": user,
            "precision": precision_u,
            "pred_items": pred_items,
            "gt_items": list(gt_items),
            "hit_items": list(hits)
        })

    mean_precision = np.mean(precisions) if precisions else 0.0
    return mean_precision, cold_start_users, per_user_result


# ================= LOAD GROUNDTRUTH =================
with open("final_groundtruth_clean.pkl", "rb") as f:
    groundtruth = pickle.load(f)

print(f"👥 Số customer trong groundtruth: {len(groundtruth)}")


# ================= LOAD SUBMISSION =================
with open("submission/submission_without_hist.json", "r", encoding="utf-8") as f:
    submission = json.load(f)

# fix key type nếu cần
try:
    submission_fixed = {int(k): v for k, v in submission.items()}
except ValueError:
    submission_fixed = submission

print(f"📁 Số customer trong submission: {len(submission_fixed)}")


# ================= EVALUATE =================
precision, cold_users, per_user = precision_at_k(
    submission_fixed,
    groundtruth,
    hist=None,
    K=10
)

print(f"\n🎯 Precision@10 (overall): {precision:.6f}")
print(f"❄️ Số cold-start users: {len(cold_users)}")


# ================= TOP 2 CUSTOMER CÓ PRECISION CAO NHẤT =================
top_users = sorted(
    per_user,
    key=lambda x: x["precision"],
    reverse=True
)[:2]

print("\n🔥 TOP 2 CUSTOMER CÓ ĐỘ CHÍNH XÁC CAO NHẤT\n")

for i, u in enumerate(top_users, 1):
    print(f"#{i}")
    print(f"Customer ID : {u['customer_id']}")
    print(f"Precision   : {u['precision']:.2f}")
    print(f"Pred items  : {u['pred_items']}")
    print(f"GT items    : {u['gt_items']}")
    print("-" * 50)
# ================= IN 1 COLD-START USER =================
top_items_df = pd.read_csv("top10_first_purchase_by_count.csv")
top_items = top_items_df["item_id"].tolist()
if cold_users:
    cold_user = cold_users[0]   # lấy 1 user bất kỳ
    print("\n🧊 VÍ DỤ 1 COLD-START USER")
    print(f"Customer ID : {cold_user}")
    print(f"GT items    : {groundtruth[cold_user]}")
    print(f"pred        : {top_items}")
    
else:
    print("\n✅ Không có cold-start user nào")



👥 Số customer trong groundtruth: 644970
📁 Số customer trong submission: 2094582

🎯 Precision@10 (overall): 0.034826
❄️ Số cold-start users: 148385

🔥 TOP 2 CUSTOMER CÓ ĐỘ CHÍNH XÁC CAO NHẤT

#1
Customer ID : 1181664
Precision   : 1.00
Pred items  : ['4603024000002', '3047000000002', '3052000000001', '3880000000001', '5444000000016', '3880000000002', '4690000000001', '2798000000001', '2803000000011', '2803000000012']
GT items    : ['6796000000002', '2798000000001', '2803000000011', '3518000000157', '5427000000006', '3880000000001', '2278000000027', '4398000000004', '7115000000005', '1370000000012', '1371000000004', '5029000000006', '1371000000005', '2278000000042', '2982000000001', '2793000000004', '7238000000001', '1370000000014', '1371000000007', '3880000000002', '2950000000002', '5444000000016', '4690000000001', '5537000000001', '4603024000002', '1371000000001', '1370000000013', '3951000000228', '5444000000017', '1974000000006', '0029140000031', '4400000000005', '2278000000039', '201